[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/02_bpe_desde_cero/02_bpe_desde_cero.ipynb)

# Módulo 02 · BPE desde cero
**Tokenización y Embeddings: de BPE a los modelos modernos**

| | |
|---|---|
| **Nivel** | 🟡 Intermedio |
| **Duración** | ≈ 5 h (lección ≈ 3 h + proyecto ≈ 2 h) |
| **GPU / coste** | Ninguna: CPU (0 unidades de Colab Pro) |
| **Prerrequisitos** | Módulo 00 (corpus Políglota) y módulo 01 (bytes, UTF-8, normalización) |

> **Estado del arte de referencia: octubre de 2026.** Las cifras sujetas a cambio llevan fecha; las técnicas se etiquetan 🟢 / 🟡 / 🔴.

## Objetivos de aprendizaje
1. **Explicar** por qué los LLM usan sub-palabras y situar BPE entre la tokenización por palabras y por caracteres.
2. **Resolver a mano** el entrenamiento de BPE sobre un corpus de 4 palabras y **reproducirlo** en código.
3. **Implementar** byte-level BPE completo: pre-tokenización con regex, entrenamiento con conteo incremental, `encode` por orden de merges y `decode`.
4. **Comparar** los patrones de pre-tokenización de GPT-2, cl100k y o200k y **predecir** cómo trocean texto, números y código.
5. **Verificar** tu implementación contra `tiktoken` y Hugging Face `tokenizers` (y contra el vocabulario real de GPT-2), y **explicar** las diferencias de entrenamiento (desempates).
6. **Medir** compresión (bytes/token, bits/byte), efecto del tamaño de vocabulario, «impuesto» por idioma y complejidad temporal.
7. **Analizar** BPE como compresor (analogía con Huffman/LZ) y sus límites teóricos.

## 🔁 Conexión con módulos anteriores

**1.** ¿Cuántos bytes UTF-8 ocupa «ñ» y qué pasa si el vocabulario no tiene un token que la contenga?

<details>
<summary>Ver respuesta</summary>

2 bytes (`C3 B1`). Un tokenizer byte-level la representa como dos tokens de un byte: nunca hay «desconocidos», pero cuesta más. Este módulo te enseña por qué.

</details>

**2.** ¿Por qué `é` en NFC y en NFD producen tokens distintos en un tokenizer byte-level?

<details>
<summary>Ver respuesta</summary>

Porque el tokenizer solo ve bytes: NFC → `C3 A9`; NFD → `65 CC 81`. Son secuencias distintas sobre las que se aprendieron (o no) fusiones distintas.

</details>

**3.** ¿Qué convención de desempates usa `rank_from_scores` en el módulo 00 y por qué importa?

<details>
<summary>Ver respuesta</summary>

Orden del corpus (determinista). Los empates arbitrarios cambian rankings entre ejecuciones; en BPE los desempates entre pares con el mismo conteo también cambian el vocabulario (lo verás hoy).

</details>

**4.** ¿Qué hace tu `normalizar` (módulo 01) y dónde encaja respecto al tokenizer?

<details>
<summary>Ver respuesta</summary>

Deja el texto en una forma canónica (NFC, sin invisibles ni mojibake…) **antes** del tokenizer y del índice; hay que aplicar la misma función en entrenamiento e inferencia.

</details>

> **🧭 Ruta núcleo vs ampliación.** La lección tiene ≈ 80 celdas. **Ruta núcleo (≈ 2 h 15 min):** secciones 1–6, 8 (verificación con tiktoken), 9.1 (tamaño de vocabulario) y 9.3 (impuesto por idioma). **Ampliación:** 7 (decodificación incremental y especiales), 9.2, 9.4–9.6 (regex, compresión, orden de merges) y 10 (complejidad). El proyecto asume la ruta núcleo.

In [ ]:
!pip install -q numpy pandas matplotlib regex tiktoken tokenizers

In [ ]:
import matplotlib.pyplot as plt

AZUL, NARANJA, VERDE, GRIS, ROJO = "#1F6FEB", "#F28E2B", "#2CA02C", "#6B7280", "#D62728"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10, "axes.titlesize": 11})

import heapq, math, random, sys, time, unicodedata, zlib, bz2, lzma, re, collections
from collections import Counter, defaultdict
import numpy as np, pandas as pd

SEED = 42
random.seed(SEED); np.random.seed(SEED)
FAST_DEV_RUN = True      # True: vocabularios de 1 000 y corpus pequeño (segundos). False: vocabularios de 4 000 y corpus mayor.
VOCAB = 1000 if FAST_DEV_RUN else 4000
print(f"python {sys.version.split()[0]} | FAST_DEV_RUN = {FAST_DEV_RUN} | VOCAB = {VOCAB}")

## 1. Intuición: tres formas de partir un texto

| Estrategia | Vocabulario | Palabras desconocidas | Longitud de la secuencia |
|---|---|---|---|
| **Palabras** | enorme (cientos de miles) y aun así incompleto | **sí** (OOV → `<unk>`) | corta |
| **Caracteres / bytes** | diminuto (≤ 256 bytes) | no | **muy larga** (1 token por byte) |
| **Sub-palabras (BPE)** | intermedio (32 000 – 262 000) | **no** (si es byte-level) | media |

BPE busca el punto medio **aprendiendo de los datos**: las secuencias frecuentes se convierten en tokens («ción», « the»), las raras se descomponen en piezas más pequeñas, y en el peor caso en bytes. Es un **compromiso entre tamaño de vocabulario y longitud de secuencia**: cada token extra en el vocabulario acorta los textos, pero cuesta una fila más en la matriz de embeddings (módulo 09) y una neurona más en la salida softmax.

> **Analogía con compresión.** BPE nació como un compresor (Gage, 1994): sustituye el par de bytes más frecuente por un símbolo nuevo y repite. Lo que en compresión se llama «diccionario» es aquí el vocabulario; «bits por byte» (sección 9.5) mide cuánto comprime.

Empezamos cargando el corpus Políglota y partiéndolo en entrenamiento (80 %) y prueba (20 %) por documento, para medir siempre con datos que el tokenizer no ha visto.

In [ ]:
# ── carga_corpus (1/3) ──
# ── Datos para el corpus SINTÉTICO de Políglota (último recurso si no hay red / Hugging Face) ──
# (artículo+concepto ES, artículo+concepto EN, definición ES, definición EN)
CONCEPTOS = [
    ("el índice invertido", "the inverted index", "una estructura que asocia cada término con la lista de documentos que lo contienen", "a structure that maps each term to the list of documents that contain it"),
    ("la caché", "the cache", "una memoria rápida que guarda resultados recientes para no recalcularlos", "a fast memory that stores recent results so they are not recomputed"),
    ("la base de datos relacional", "the relational database", "un sistema que organiza los datos en tablas relacionadas mediante claves", "a system that organizes data into tables related through keys"),
    ("el balanceador de carga", "the load balancer", "un componente que reparte las peticiones entre varios servidores", "a component that spreads requests across several servers"),
    ("el contenedor Docker", "the Docker container", "un proceso aislado que empaqueta una aplicación con sus dependencias", "an isolated process that packages an application with its dependencies"),
    ("el control de versiones", "version control", "un sistema que registra los cambios de los archivos a lo largo del tiempo", "a system that records changes to files over time"),
    ("la API REST", "the REST API", "una interfaz que expone recursos mediante URL y verbos HTTP", "an interface that exposes resources through URLs and HTTP verbs"),
    ("la autenticación con tokens", "token authentication", "un mecanismo que identifica al usuario con un testigo firmado en cada petición", "a mechanism that identifies the user with a signed token on every request"),
    ("la expresión regular", "the regular expression", "un patrón que describe un conjunto de cadenas de texto", "a pattern that describes a set of text strings"),
    ("el hilo de ejecución", "the thread", "una secuencia de instrucciones que comparte memoria con otros hilos del proceso", "a sequence of instructions that shares memory with other threads of the process"),
    ("el recolector de basura", "the garbage collector", "un mecanismo que libera automáticamente la memoria que ya no se usa", "a mechanism that automatically frees memory that is no longer used"),
    ("la cola de mensajes", "the message queue", "un búfer que desacopla a los productores de los consumidores de eventos", "a buffer that decouples event producers from consumers"),
    ("la tabla hash", "the hash table", "una estructura que guarda pares clave-valor y busca en tiempo casi constante", "a structure that stores key-value pairs and looks them up in near-constant time"),
    ("el árbol binario de búsqueda", "the binary search tree", "un árbol ordenado donde cada nodo separa los valores menores de los mayores", "an ordered tree where each node separates smaller values from larger ones"),
    ("la red neuronal", "the neural network", "un modelo formado por capas de unidades que aprenden pesos a partir de ejemplos", "a model made of layers of units that learn weights from examples"),
    ("el descenso de gradiente", "gradient descent", "un método que ajusta los parámetros en la dirección que reduce la pérdida", "a method that adjusts parameters in the direction that reduces the loss"),
    ("el sobreajuste", "overfitting", "el fenómeno por el que un modelo memoriza el entrenamiento y generaliza mal", "the phenomenon where a model memorizes the training set and generalizes poorly"),
    ("la validación cruzada", "cross-validation", "una técnica que estima el rendimiento repartiendo los datos en varios pliegues", "a technique that estimates performance by splitting the data into several folds"),
    ("el tokenizador", "the tokenizer", "el componente que convierte el texto en una secuencia de identificadores enteros", "the component that turns text into a sequence of integer identifiers"),
    ("la ventana de contexto", "the context window", "la cantidad máxima de tokens que un modelo puede leer a la vez", "the maximum number of tokens a model can read at once"),
    ("el compilador", "the compiler", "un programa que traduce código fuente a instrucciones de máquina", "a program that translates source code into machine instructions"),
    ("el sistema de archivos", "the file system", "la capa que organiza cómo se guardan y se nombran los datos en disco", "the layer that organizes how data is stored and named on disk"),
    ("la memoria virtual", "virtual memory", "una abstracción que da a cada proceso su propio espacio de direcciones", "an abstraction that gives each process its own address space"),
    ("el protocolo TCP", "the TCP protocol", "un protocolo que entrega bytes de forma ordenada y fiable entre dos máquinas", "a protocol that delivers bytes reliably and in order between two machines"),
    ("el cifrado simétrico", "symmetric encryption", "un esquema que usa la misma clave para cifrar y descifrar", "a scheme that uses the same key to encrypt and decrypt"),
    ("las pruebas unitarias", "unit tests", "comprobaciones automáticas que verifican una función aislada", "automatic checks that verify a single isolated function"),
    ("la integración continua", "continuous integration", "una práctica que compila y prueba cada cambio de forma automática", "a practice that builds and tests every change automatically"),
    ("los microservicios", "microservices", "una arquitectura que divide la aplicación en servicios pequeños e independientes", "an architecture that splits the application into small independent services"),
    ("la paginación", "pagination", "una técnica que devuelve los resultados en bloques de tamaño fijo", "a technique that returns results in fixed-size blocks"),
    ("la limitación de tasa", "rate limiting", "una política que restringe cuántas peticiones puede hacer un cliente por minuto", "a policy that restricts how many requests a client can make per minute"),
    ("la serialización JSON", "JSON serialization", "el proceso de convertir objetos en texto JSON y viceversa", "the process of converting objects to JSON text and back"),
    ("la variable de entorno", "the environment variable", "un par nombre-valor que el sistema operativo entrega a cada proceso", "a name-value pair that the operating system hands to each process"),
    ("el entorno virtual", "the virtual environment", "una carpeta aislada con su propio intérprete de Python y sus paquetes", "an isolated folder with its own Python interpreter and packages"),
    ("el decorador", "the decorator", "una función que envuelve a otra para añadirle comportamiento", "a function that wraps another one to add behavior"),
    ("el generador", "the generator", "una función que produce valores perezosamente con la palabra yield", "a function that lazily produces values with the yield keyword"),
    ("el manejo de excepciones", "exception handling", "el mecanismo para capturar errores sin que el programa se detenga", "the mechanism for catching errors without stopping the program"),
    ("la programación asíncrona", "asynchronous programming", "un modelo que solapa esperas de entrada y salida sin bloquear el hilo", "a model that overlaps input and output waits without blocking the thread"),
    ("el bloqueo de base de datos", "the database lock", "un candado que impide que dos transacciones modifiquen la misma fila a la vez", "a lock that prevents two transactions from modifying the same row at once"),
    ("la migración de esquema", "the schema migration", "un script versionado que cambia la estructura de la base de datos", "a versioned script that changes the structure of the database"),
    ("el registro de eventos", "event logging", "la práctica de escribir mensajes con marca de tiempo sobre lo que hace el sistema", "the practice of writing timestamped messages about what the system does"),
]
HERRAMIENTAS = ["PostgreSQL", "Redis", "nginx", "Docker", "pytest", "Kubernetes", "FastAPI", "SQLite", "Git", "Linux", "Celery", "Prometheus"]
PARAMETROS = ["max_connections", "timeout_seconds", "cache_size", "batch_size", "log_level", "retry_count", "pool_size", "num_workers"]
# aspecto -> (título ES, título EN, palabra clave ES, palabra clave EN, plantillas ES, plantillas EN)
ASPECTOS = {
    "intro": ("{n}: qué es y para qué sirve", "{n}: what it is and what it is for", "qué es", "what is",
        ["Se usa en {t} para simplificar el trabajo diario del equipo.", "Entender {c} es un requisito antes de pasar a producción.", "Su idea central es sencilla y aparece en casi todos los sistemas modernos.", "La documentación de {t} dedica un capítulo completo a {c}."],
        ["It is used in {t} to simplify the daily work of the team.", "Understanding {c} is a prerequisite before going to production.", "Its core idea is simple and appears in almost every modern system.", "The {t} documentation devotes a whole chapter to {c}."]),
    "config": ("{n}: cómo configurarlo", "{n}: how to configure it", "configurar", "configure",
        ["Para configurar {c} edita el archivo de ajustes y reinicia {t}.", "El parámetro {p} controla el comportamiento de {c} y su valor por defecto es {x}.", "Después de cambiar {p} a {x}, comprueba el resultado con un comando de diagnóstico.", "Guarda la configuración de {c} en un archivo versionado y no en la consola."],
        ["To configure {c}, edit the settings file and restart {t}.", "The {p} parameter controls the behavior of {c} and its default value is {x}.", "After changing {p} to {x}, check the result with a diagnostic command.", "Keep the configuration of {c} in a versioned file and not in the console."]),
    "errores": ("{n}: errores frecuentes", "{n}: common errors", "errores", "errors",
        ["Un error frecuente con {c} es olvidar el valor de {p} y dejarlo en {x}.", "Si aparece un fallo intermitente en {t}, revisa primero los registros de {c}.", "Muchos equipos descubren el problema de {c} solo cuando la carga se multiplica por {x}.", "El mensaje de error suele ser engañoso: la causa real está en {c}, no en {t}."],
        ["A common mistake with {c} is forgetting {p} and leaving it at {x}.", "If an intermittent failure shows up in {t}, check the logs of {c} first.", "Many teams only discover the problem with {c} when the load grows by {x} times.", "The error message is usually misleading: the real cause is in {c}, not in {t}."]),
    "rendimiento": ("{n}: rendimiento y optimización", "{n}: performance and optimization", "rendimiento", "performance",
        ["Para medir el rendimiento de {c} usa un perfilador y no tu intuición.", "Subir {p} a {x} mejora la latencia, pero consume más memoria en {t}.", "La optimización prematura de {c} suele complicar el código sin ganar tiempo.", "En una prueba con {x} usuarios, {c} redujo la latencia mediana a la mitad."],
        ["To measure the performance of {c}, use a profiler and not your intuition.", "Raising {p} to {x} improves latency but uses more memory in {t}.", "Premature optimization of {c} usually complicates the code without saving time.", "In a test with {x} users, {c} cut the median latency in half."]),
    "practicas": ("{n}: buenas prácticas", "{n}: best practices", "buenas prácticas", "best practices",
        ["Una buena práctica con {c} es documentar {p} y automatizar su despliegue en {t}.", "Revisa {c} en cada revisión de código y escribe pruebas para los casos límite.", "Mantén {c} simple: añade complejidad solo cuando los datos lo justifiquen.", "Define alertas sobre {c} antes del primer incidente, no después."],
        ["A good practice with {c} is to document {p} and automate its deployment in {t}.", "Review {c} in every code review and write tests for the edge cases.", "Keep {c} simple: add complexity only when the data justifies it.", "Define alerts on {c} before the first incident, not after."]),
    "ejemplo": ("{n}: ejemplo en Python", "{n}: Python example", "ejemplo", "example",
        ["En este ejemplo en Python usamos {t} para mostrar {c} paso a paso.", "El script define {p} = {x} y luego imprime el resultado de {c}.", "Ejecuta el ejemplo con Python 3.11 y compara la salida con la esperada.", "Puedes adaptar el código de {c} a tu propio proyecto cambiando solo {p}."],
        ["In this Python example we use {t} to show {c} step by step.", "The script defines {p} = {x} and then prints the result of {c}.", "Run the example with Python 3.11 and compare the output with the expected one.", "You can adapt the code for {c} to your own project by changing only {p}."]),
    "seguridad": ("{n}: consideraciones de seguridad", "{n}: security considerations", "seguridad", "security",
        ["Desde el punto de vista de la seguridad, {c} nunca debe exponerse sin autenticación.", "Rota las credenciales de {t} cada {x} días y no las guardes en el repositorio.", "Un ataque contra {c} suele aprovechar el valor de {p} mal configurado.", "Audita los accesos a {c} y conserva los registros durante {x} días."],
        ["From a security standpoint, {c} must never be exposed without authentication.", "Rotate the {t} credentials every {x} days and do not store them in the repository.", "An attack on {c} usually exploits a misconfigured {p} value.", "Audit access to {c} and keep the logs for {x} days."]),
}

In [ ]:
# ── carga_corpus (2/3) ──
# ── Generador del corpus sintético (mismo esquema que el real) ────────────────
import random


def _es_fix(s: str) -> str:
    """Contracciones del español: 'a el' -> 'al', 'de el' -> 'del'."""
    return s.replace(" a el ", " al ").replace(" de el ", " del ").replace("De el ", "Del ").replace("A el ", "Al ")


def _sintetico_wiki(lang: str, n_docs: int, n_queries: int, seed: int = 42):
    """Docs técnicos sintéticos ES/EN: concepto x aspecto x herramienta. Devuelve (docs, queries, qrels)."""
    rng = random.Random(seed + (0 if lang == "es" else 1))
    li = 0 if lang == "es" else 1
    combos = [(c, a, t) for c in range(len(CONCEPTOS)) for a in ASPECTOS for t in range(len(HERRAMIENTAS))]
    rng.shuffle(combos)
    docs, titles = [], []
    for k, (ci, asp, ti) in enumerate(combos[:n_docs]):
        concepto, definicion = CONCEPTOS[ci][li], CONCEPTOS[ci][2 + li]
        tool = HERRAMIENTAS[ti]
        tit_t, plantillas = ASPECTOS[asp][li], ASPECTOS[asp][4 + li]
        nombre = concepto.split(" ", 1)[1] if concepto.split(" ", 1)[0] in ("el", "la", "los", "las", "the") else concepto
        slots = lambda: dict(c=concepto, t=tool, p=rng.choice(PARAMETROS), x=rng.choice([8, 16, 30, 64, 100, 256, 1024]))
        frases = [("{C} " + ("es " if li == 0 else "is ") + definicion + ".").format(C=concepto)]
        frases += [pl.format(**slots()) for pl in rng.sample(plantillas, len(plantillas))]
        frases.append(rng.choice(ASPECTOS["intro"][4 + li]).format(**slots()))
        for _ in range(2):  # ruido: frases de otro concepto (hace el retrieval no trivial)
            otro = rng.randrange(len(CONCEPTOS))
            otra = rng.choice(list(ASPECTOS.values()))[4 + li]
            s = slots(); s["c"] = CONCEPTOS[otro][li]
            frases.append(rng.choice(otra).format(**s))
        s = slots()
        frases.append(("Ejemplo: `{p} = {x}`." if li == 0 else "Example: `{p} = {x}`.").format(**s))
        text = " ".join(f[0].upper() + f[1:] for f in frases)
        docs.append({"id": f"{lang}-syn-{k:04d}", "lang": lang, "text": _es_fix(text) if lang == "es" else text})
        titles.append((nombre, ASPECTOS[asp][2 + li], tool, tit_t.format(n=nombre) + f" ({tool})"))
    queries, qrels = [], {}
    for j, k in enumerate(rng.sample(range(len(docs)), min(n_queries, len(docs)))):
        nombre, kw, tool, titulo = titles[k]
        q = titulo.replace(":", "").replace("(", "").replace(")", "") if j % 2 == 0 else f"{nombre} {kw} {tool}"
        qid = f"{lang}-q{j:04d}"
        queries.append({"qid": qid, "lang": lang, "text": q})
        qrels[qid] = {docs[k]["id"]: 1}
    return docs, queries, qrels

In [ ]:
# ── carga_corpus (3/3) ──
# ── carga_corpus(): cadena oficial (HF) -> espejo (API de Wikipedia / stdlib) -> sintético ─────
import ast
import json
import os
import random
import urllib.parse
import urllib.request

SEED = 42


def _strip_docstring(func_src: str):
    """Devuelve (código sin docstring, primera frase del docstring) o None si no hay docstring útil."""
    try:
        fn = ast.parse(func_src).body[0]
        doc = ast.get_docstring(fn)
        first = fn.body[0]
    except Exception:
        return None
    if not doc or not (isinstance(first, ast.Expr) and isinstance(getattr(first, "value", None), ast.Constant)):
        return None
    lines = func_src.splitlines()
    code = "\n".join(lines[: first.lineno - 1] + lines[first.end_lineno :])
    frase = doc.strip().split("\n\n")[0].replace("\n", " ").strip()
    frase = frase.split(". ")[0].rstrip(".")
    return (code, frase) if len(frase.split()) >= 4 else None


def _code_stdlib(n_docs: int, n_queries: int, rng: random.Random):
    """Espejo OFFLINE con código real: funciones de la biblioteca estándar con su docstring como query."""
    import importlib, inspect, textwrap
    mods = ["argparse", "bisect", "calendar", "collections", "colorsys", "copy", "csv", "difflib", "fnmatch", "fractions",
            "functools", "getpass", "glob", "heapq", "html", "json", "operator", "os", "pathlib", "posixpath", "queue",
            "random", "re", "shlex", "shutil", "statistics", "string", "tempfile", "textwrap", "urllib.parse", "uuid", "zipfile"]
    cands = []
    for name in mods:
        try:
            src = inspect.getsource(importlib.import_module(name))
            tree = ast.parse(src)
        except Exception:
            continue
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.end_lineno - node.lineno >= 4:
                seg = textwrap.dedent("\n".join(src.splitlines()[node.lineno - 1 : node.end_lineno]))
                r = _strip_docstring(seg)
                if r and 200 <= len(r[0]) <= 2500:
                    cands.append((f"{name}.{node.name}", r[0], r[1]))
    rng.shuffle(cands)
    return cands[:n_docs]


def _code_hf(n_docs: int, rng: random.Random):
    """Oficial: CodeSearchNet (Python) en streaming. Prueba varios repos por cambios de formato en `datasets`."""
    from datasets import load_dataset
    errs = []
    for repo, kw in [("code-search-net/code_search_net", {"name": "python"}), ("code_search_net", {"name": "python"}),
                     ("claudios/code_search_net", {"name": "python"}), ("Nan-Do/code-search-net-python", {})]:
        try:
            ds = load_dataset(repo, split="train", streaming=True, **kw).shuffle(seed=SEED, buffer_size=2000)
            out = []
            for ex in ds:
                src = ex.get("func_code_string") or ex.get("whole_func_string") or ex.get("code") or ""
                r = _strip_docstring(src) if '"""' in src or "'''" in src else None
                doc = (ex.get("func_documentation_string") or ex.get("docstring") or "").strip().split("\n\n")[0]
                doc = doc.replace("\n", " ").split(". ")[0].rstrip(".")
                code = r[0] if r else src
                if 200 <= len(code) <= 2500 and len(doc.split()) >= 4:
                    out.append((ex.get("func_name") or f"csn.{len(out)}", code, doc))
                if len(out) >= n_docs:
                    return out
        except Exception as e:  # noqa: BLE001
            errs.append(f"{repo}: {type(e).__name__}")
    raise RuntimeError("CodeSearchNet no disponible -> " + "; ".join(errs))


def _wiki_hf(lang: str, n_docs: int, cap: int):
    """Oficial: wikimedia/wikipedia (snapshot 20231101) en streaming con un buffer pequeño."""
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", f"20231101.{lang}", split="train", streaming=True)
    ds = ds.shuffle(seed=SEED, buffer_size=2000)
    out = []
    for ex in ds:
        if len(ex["text"]) >= 600 and len(ex["title"]) >= 3:
            out.append((f"{lang}-{ex['id']}", ex["title"], ex["text"][:cap]))
        if len(out) >= n_docs:
            break
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out


def _wiki_api(lang: str, n_docs: int):
    """Espejo: API pública de Wikipedia (artículos aleatorios, solo la sección introductoria; ~20 por petición)."""
    out, seen, tries = [], set(), 0
    while len(out) < n_docs and tries < 80:
        tries += 1
        qs = urllib.parse.urlencode({"action": "query", "format": "json", "generator": "random", "grnnamespace": 0,
                                     "grnlimit": 20, "prop": "extracts", "exintro": 1, "explaintext": 1, "exlimit": 20})
        req = urllib.request.Request(f"https://{lang}.wikipedia.org/w/api.php?{qs}",
                                     headers={"User-Agent": "poliglota-course/1.0 (curso educativo)"})
        with urllib.request.urlopen(req, timeout=20) as r:
            pages = json.load(r).get("query", {}).get("pages", {})
        for p in pages.values():
            if p["pageid"] not in seen and len(p.get("extract", "")) >= 300:
                seen.add(p["pageid"])
                out.append((f"{lang}-{p['pageid']}", p["title"], p["extract"]))
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out[:n_docs]


def carga_corpus(fast_dev: bool = True) -> dict:
    """Corpus Políglota -> {"docs": [{id, lang, text}], "queries": [{qid, lang, text}], "qrels": {qid: {doc_id: 1}},
    "source": "real" | "synthetic"}. Cadena: oficial (HF) -> espejo (API Wikipedia; código de la stdlib) -> sintético."""
    rng = random.Random(SEED)
    n_wiki, n_code, q_wiki, q_code, cap = (150, 100, 40, 30, 1500) if fast_dev else (3000, 1500, 200, 150, 6000)
    docs, queries, qrels, source = [], [], {}, "real"
    wiki = {}
    for lang in ("es", "en"):
        for nombre, fn in (("HF wikimedia/wikipedia", lambda: _wiki_hf(lang, n_wiki, cap)),
                           ("API de Wikipedia (espejo)", lambda: _wiki_api(lang, min(n_wiki, 400)))):
            try:
                wiki[lang] = fn()
                print(f"[carga_corpus] Wikipedia {lang}: {len(wiki[lang])} artículos desde {nombre}")
                break
            except Exception as e:  # noqa: BLE001
                print(f"[carga_corpus] {nombre} ({lang}) falló: {type(e).__name__}: {str(e)[:80]}")
    if len(wiki) == 2:
        for lang, arts in wiki.items():
            docs += [{"id": i, "lang": lang, "text": t} for i, _, t in arts]
            for j, k in enumerate(rng.sample(range(len(arts)), min(q_wiki, len(arts)))):
                qid = f"{lang}-q{j:04d}"
                queries.append({"qid": qid, "lang": lang, "text": arts[k][1]})
                qrels[qid] = {arts[k][0]: 1}
    else:
        source = "synthetic"
        print("⚠️  [carga_corpus] SIN acceso a Wikipedia: uso texto ES/EN SINTÉTICO (mismo esquema). Los números NO son comparables con datos reales.")
        for lang in ("es", "en"):
            d, q, r = _sintetico_wiki(lang, n_wiki, q_wiki, SEED)
            docs += d; queries += q; qrels.update(r)
    try:
        code = _code_hf(n_code, rng)
        print(f"[carga_corpus] código: {len(code)} funciones desde CodeSearchNet (HF)")
    except Exception as e:  # noqa: BLE001
        print(f"[carga_corpus] CodeSearchNet falló ({str(e)[:90]}); uso funciones REALES de la stdlib de Python (espejo offline)")
        code = _code_stdlib(n_code, q_code, rng)
    for k, (name, src, doc) in enumerate(code):
        docs.append({"id": f"code-{k:04d}", "lang": "code", "text": src})
    for j, k in enumerate(rng.sample(range(len(code)), min(q_code, len(code)))):
        qid = f"code-q{j:04d}"
        queries.append({"qid": qid, "lang": "code", "text": code[k][2]})
        qrels[qid] = {f"code-{k:04d}": 1}
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": source}

In [ ]:
corpus = carga_corpus(fast_dev=FAST_DEV_RUN)
docs, source = corpus["docs"], corpus["source"]
train = {l: [d["text"] for i, d in enumerate([d for d in docs if d["lang"] == l]) if i % 5 != 0] for l in ("es", "en", "code")}
test = {l: [d["text"] for i, d in enumerate([d for d in docs if d["lang"] == l]) if i % 5 == 0] for l in ("es", "en", "code")}
nbytes = lambda ts: sum(len(t.encode("utf-8")) for t in ts)
print(f"source = {source!r}")
print(pd.DataFrame({"train (KB)": {l: nbytes(v) // 1000 for l, v in train.items()}, "test (KB)": {l: nbytes(v) // 1000 for l, v in test.items()}}))

In [ ]:
# Cobertura: palabras (OOV) vs caracteres vs BPE. Un vocabulario de palabras de tamaño K cubre el texto de prueba… ¿en qué %?
tok_pal = lambda t: re.findall(r"\w+|[^\w\s]", t.lower())
filas = []
for l in ("es", "en", "code"):
    frec = Counter(w for t in train[l] for w in tok_pal(t))
    prueba = [w for t in test[l] for w in tok_pal(t)]
    for K in (100, 300, 1000, 3000):
        top = {w for w, _ in frec.most_common(K)}
        filas.append({"idioma": l, "K (palabras en el vocabulario)": K, "OOV %": 100 * np.mean([w not in top for w in prueba])})
df_oov = pd.DataFrame(filas)
fig, ax = plt.subplots(figsize=(7, 3.6))
for l, c in zip(("es", "en", "code"), (AZUL, NARANJA, VERDE)):
    d = df_oov[df_oov["idioma"] == l]; ax.plot(d["K (palabras en el vocabulario)"], d["OOV %"], "o-", color=c, label=l)
ax.set_xscale("log"); ax.set_xlabel("tamaño del vocabulario de palabras K"); ax.set_ylabel("% de palabras de prueba fuera del vocabulario")
ax.set_title(f"Un vocabulario de palabras siempre deja OOV (corpus {source})"); ax.legend(); plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** Las curvas bajan al crecer K pero **nunca llegan a 0** (con texto real, el «ruido» de nombres, números, errores y palabras raras es infinito): ahí un vocabulario de palabras necesita `<unk>`. BPE a nivel de bytes resuelve el problema de raíz: el vocabulario base son los 256 bytes y **cualquier** cadena es codificable. Con el corpus sintético las curvas caen más rápido que con Wikipedia real (generador repetitivo).

> ### 🎨 GEMINI-IMAGEN · ID: `M02-IMG-01` · ⚠️ REEMPLAZAR ESTA CELDA
> **Pipeline de un tokenizer byte-level BPE**
>
> **Qué muestra:** El recorrido completo de un texto por un tokenizer estilo GPT-2: regex de pre-tokenización, bytes, aplicación de merges por rango, ids; y la dirección inversa (decode).
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Pipeline de byte-level BPE: texto, regex, bytes, merges por rango, ids y decodificación inversa
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio de un pipeline, fondo blanco, 16:9, flechas gruesas de izquierda a derecha. Paleta: azul #1F6FEB para texto y tokens, naranja #F28E2B para reglas y tablas de merges, verde #2CA02C para ids/resultado, gris #6B7280 para anotaciones. Tipografía sans-serif legible y monoespaciada para bytes, sin ilustraciones decorativas.
> Composición: 5 cajas en fila con título arriba y contenido debajo. Textos EXACTOS:
> 1) «Texto»: «Hello, world!».
> 2) «Pre-tokenización (regex)»: cuatro cajitas azules «Hello», «,», « world», «!» con la nota «el espacio se queda pegado a la palabra siguiente».
> 3) «Bytes UTF-8»: bajo cada trozo sus bytes en hexadecimal: «48 65 6C 6C 6F», «2C», «20 77 6F 72 6C 64», «21».
> 4) «Merges por rango (menor rango primero)»: una pequeña tabla naranja con el título «tabla de merges (par → token nuevo)» y las filas «H e → He», «l l → ll», «He ll → Hell», «Hell o → Hello»; abajo, la leyenda «se aplica siempre el par aprendido antes» y la nota «ejemplo ilustrativo: no son los merges reales de GPT-2».
> 5) «Ids»: cuatro números verdes EXACTOS en este orden: «15496», «11», «995», «0» con la etiqueta «vocabulario GPT-2».
> Debajo de las cajas, una flecha de derecha a izquierda de color gris etiquetada «decode: ids → bytes → texto (UTF-8)».
> Título superior: «Byte-level BPE: de texto a ids».
> Evita: números distintos de los indicados, tokens inventados, texto ilegible, flechas que se crucen, elementos decorativos.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M02-IMG-01.png` en la carpeta del módulo y reemplaza esta celda por: `![Pipeline de byte-level BPE: texto, regex, bytes, merges por rango, ids y decodificación inversa](assets/M02-IMG-01.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> "Hello, world!" → regex → [Hello][,][ world][!] → bytes 48 65 6C 6C 6F | 2C | 20 77 6F 72 6C 64 | 21
>    → merges por rango: H e→He, l l→ll, He ll→Hell, Hell o→Hello
>    → ids (GPT-2): 15496 | 11 | 995 | 0           ← decode: ids → bytes → UTF-8
> ```

> 👀 **Qué debes observar.** Sigue una cadena: el regex **parte primero**, los merges **solo se aplican dentro de cada trozo**. Los ids `15496, 11, 995, 0` son exactamente los que devuelve `tiktoken` con el vocabulario de GPT-2 (lo comprobarás en la sección 8). La flecha inversa (decode) es trivial: concatenar bytes.

## 2. Historia y fuentes (qué es de quién)
| Año | Aporte | Fuente |
|---|---|---|
| 1994 | **Gage**: BPE como compresor de datos (sustituir el par de bytes más frecuente por un byte libre) | *A New Algorithm for Data Compression*, C Users Journal 12(2) |
| 2016 | **Sennrich, Haddow & Birch**: BPE para *segmentación en sub-palabras* en traducción neuronal; entrenan sobre un diccionario de palabras ponderado por frecuencia, sin cruzar fronteras de palabra, y fijan el nº de merges como único hiperparámetro | arXiv:1508.07909 |
| 2019 | **Wang, Cho & Gu**: BPE sobre **bytes** (BBPE): vocabulario base de 256, sin OOV | arXiv:1909.03341 |
| 2019 | **GPT-2** (Radford et al.): byte-level BPE + **regex de pre-tokenización** (no fusionar entre categorías de caracteres, salvo el espacio) y vocabulario de 50 257 | informe técnico de OpenAI |
| 2023 | `tiktoken` (OpenAI): implementación rápida; encodings `cl100k_base` y `o200k_base` | repositorio `openai/tiktoken` |
| 2024 | Llama 3 usa un vocabulario de 128 K: 100 K de `tiktoken` + 28 K tokens adicionales para idiomas no ingleses | Llama Team 2024, arXiv:2407.21783 |

## 3. Ejemplo resuelto a mano (el clásico de Sennrich)

Antes de la fórmula, ejecútalo a mano: la regla es *«cuenta los pares de símbolos adyacentes, fusiona el más frecuente, repite»*. Es el mismo espíritu que un compresor de diccionario (LZW): sustituir lo que se repite por un símbolo nuevo.

Corpus: «low» ×5, «lower» ×2, «newest» ×6, «widest» ×3. Cada palabra termina con un marcador `</w>` y empieza como secuencia de caracteres. (Es el diccionario del Algoritmo 1 de Sennrich et al., 2016.)

| Palabra | Frecuencia | Estado inicial |
|---|---|---|
| low | 5 | `l o w </w>` |
| lower | 2 | `l o w e r </w>` |
| newest | 6 | `n e w e s t </w>` |
| widest | 3 | `w i d e s t </w>` |

**Paso 1.** Pares y conteos: `e s` = 6 (newest) + 3 (widest) = **9**; `s t` = 9; `t </w>` = 9; `w e` = 2 + 6 = 8; `l o` = 5 + 2 = 7; `o w` = 7; … Hay un **empate a 9**. Convención del curso: **mayor conteo; ante empate, el par menor lexicográficamente** → fusionamos `e s` → `es`.
**Paso 2.** Ahora `es t` = 9 y `t </w>` = 9 → `es t` → `est`. **Paso 3.** `est </w>` = 9 → `est</w>`. **Paso 4.** `l o` = 7 = `o w` = 7 → `l o` → `lo`. **Paso 5.** `lo w` = 7 → `low`.

Tras 5 fusiones: `low </w>`×5, `low e r </w>`×2, `n e w est</w>`×6, `w i d est</w>`×3. Lo comprobamos con código.

In [ ]:
def stats_pares(vocab):
    c = Counter()
    for w, f in vocab.items():
        for p in zip(w, w[1:]): c[p] += f
    return c

def fusiona_vocab(par, vocab):
    nuevo = {}
    for w, f in vocab.items():
        out, i = [], 0
        while i < len(w):
            if i < len(w) - 1 and (w[i], w[i + 1]) == par: out.append(w[i] + w[i + 1]); i += 2
            else: out.append(w[i]); i += 1
        nuevo[tuple(out)] = f
    return nuevo

vocab = {("l", "o", "w", "</w>"): 5, ("l", "o", "w", "e", "r", "</w>"): 2, ("n", "e", "w", "e", "s", "t", "</w>"): 6, ("w", "i", "d", "e", "s", "t", "</w>"): 3}
filas = []
for paso in range(1, 11):
    c = stats_pares(vocab)
    mejor = min(c.values(), key=lambda v: -v); empatados = sorted(p for p, v in c.items() if v == mejor)
    par = empatados[0]; vocab = fusiona_vocab(par, vocab)
    filas.append({"paso": paso, "par fusionado": " + ".join(par), "conteo": mejor, "empatados": len(empatados),
                  "estado": " | ".join(f"{' '.join(w)}×{f}" for w, f in vocab.items())})
pd.set_option("display.max_colwidth", 120)
display(pd.DataFrame(filas).set_index("paso"))
assert [f["par fusionado"] for f in filas[:5]] == ["e + s", "es + t", "est + </w>", "l + o", "lo + w"] and [f["conteo"] for f in filas[:5]] == [9, 9, 9, 7, 7]

> ### 🎬 GEMINI-VIDEO · ID: `M02-VID-01` · ⚠️ REEMPLAZAR ESTA CELDA
> **Animación: BPE fusiona el par más frecuente**
>
> **Qué muestra:** El proceso de merges del ejemplo clásico (low, lower, newest, widest), paso a paso, con la barra de conteos de pares y el vocabulario que crece. Es el proceso dinámico central del módulo.
> **Formato:** MP4 16:9, 1080p | Duración: 30 s | **Texto alternativo:** Animación de 30 segundos con las cinco primeras fusiones de BPE sobre low, lower, newest y widest
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Video explicativo de 16:9, ritmo calmado, sin voz en off, con música ausente. Fondo blanco, estilo de diagrama técnico limpio, paleta: azul #1F6FEB para símbolos, naranja #F28E2B para el par que se fusiona, verde #2CA02C para el token nuevo, gris #6B7280 para anotaciones. Tipografía monoespaciada para los símbolos y sans-serif para los textos. Sin ilustraciones decorativas.
> Guion por escenas (textos en pantalla EXACTOS):
> 0–4 s: Título «Byte Pair Encoding: fusionar el par más frecuente». Aparece una tabla de cuatro filas con la palabra, su frecuencia y su secuencia de símbolos: «low ×5 → l o w </w>», «lower ×2 → l o w e r </w>», «newest ×6 → n e w e s t </w>», «widest ×3 → w i d e s t </w>».
> 4–9 s: A la derecha aparece un gráfico de barras horizontales «Conteo de pares» con: «e s: 9», «s t: 9», «t </w>: 9», «w e: 8», «l o: 7», «o w: 7». Las tres barras de 9 parpadean en naranja con la nota «empate: se elige el menor lexicográficamente».
> 9–13 s: Paso 1. Las «e s» de las palabras newest y widest se resaltan en naranja y se funden en una sola caja verde «es». Texto: «Paso 1: e + s → es (9)».
> 13–16 s: Paso 2: «es + t → est (9)»; las cajas «es» y «t» se funden en «est».
> 16–19 s: Paso 3: «est + </w> → est</w> (9)».
> 19–22 s: Paso 4: «l + o → lo (7)».
> 22–25 s: Paso 5: «lo + w → low (7)».
> 25–30 s: Estado final en la tabla, EXACTO: «low </w> ×5», «low e r </w> ×2», «n e w est</w> ×6», «w i d est</w> ×3». Una lista lateral «Vocabulario aprendido» muestra en orden: «es», «est», «est</w>», «lo», «low». Texto final: «Cada fusión acorta el corpus y añade un token al vocabulario».
> Evita: cambiar números o pares, texto ilegible, movimientos bruscos, elementos decorativos, mostrar pares distintos de los indicados.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M02-VID-01.mp4` en la carpeta del módulo y reemplaza esta celda por: `<video src="assets/M02-VID-01.mp4" controls width="100%"></video>`
>
> **Mientras tanto (placeholder):**
> ```text
> t=0   corpus:  low×5 | lower×2 | newest×6 | widest×3  (cada palabra = caracteres + </w>)
> t=4   pares:   e s:9  s t:9  t </w>:9  w e:8  l o:7  o w:7     (empate a 9)
> t=9   1) e+s→es   2) es+t→est   3) est+</w>→est</w>   4) l+o→lo   5) lo+w→low
> t=25  final:   low </w>×5 | low e r </w>×2 | n e w est</w>×6 | w i d est</w>×3
>        vocabulario aprendido: es, est, est</w>, lo, low
> ```

> 👀 **Qué debes observar.** Fíjate en cómo cambian los conteos tras cada fusión: al fusionar `e s`, el par `s t` **desaparece** (ahora la `s` está dentro de `es`) y nace `es t` con el mismo conteo. Eso es lo que hace costoso el algoritmo ingenuo (recontar todo) y lo que explota el conteo incremental de la sección 6: solo cambian los pares **vecinos** de la fusión.

## 4. El algoritmo, formalmente

Acabas de ver el algoritmo funcionar con 16 apariciones de 4 palabras; ahora lo escribimos en general.

Sea el corpus una secuencia (o un multiconjunto de «palabras» con frecuencia) sobre un alfabeto inicial $\Sigma_0$ (aquí, los 256 bytes). **Entrenamiento**, con $M$ fusiones:

$$\text{para } k=1,\dots,M:\quad (a_k,b_k)=\arg\max_{(a,b)}\ \text{count}(a,b);\qquad t_k := a_k b_k;\qquad \Sigma_k=\Sigma_{k-1}\cup\{t_k\}$$

y se reemplaza cada aparición adyacente de $(a_k,b_k)$ por $t_k$ en el corpus. Aquí $\text{count}(a,b)$ es el número de veces que $b$ sigue a $a$ en el corpus **actual**, ponderado por la frecuencia de cada palabra. El tamaño final del vocabulario es $|V|=256+M$ (más tokens especiales).

**Codificación** de un texto nuevo: se parte en trozos (pre-tokenización), cada trozo se convierte a bytes y se aplican las fusiones **en el orden en que se aprendieron**: en cada paso se fusiona el par adyacente cuyo token resultante tiene **menor rango** $k$, hasta que no queda ninguno. El **orden de los merges es parte del tokenizer**: dos listas con las mismas fusiones en distinto orden son tokenizers distintos.

**¿Qué optimiza?** Cada fusión de $(a,b)$ reduce la longitud del corpus en (≈) $\text{count}(a,b)$ tokens, así que BPE es un **algoritmo voraz de compresión**. Formalizar *qué* problema resuelve costó casi 30 años: Zouhar et al. (2023) lo plantean como un problema de optimización combinatoria y prueban que la versión voraz es una aproximación de factor $\tfrac{1}{\sigma}(1-e^{-\sigma})$ (≈ 0.37 como cota empírica) del óptimo; Kozma & Voderholzer (2024) demuestran que encontrar la secuencia óptima de $M$ fusiones es **APX-completo** y que BPE garantiza entre 0.333 y 0.625 del óptimo en el peor caso.

> ⚠️ **Más compresión ≠ mejor modelo.** Schmidt et al. (2024) no encuentran que menos tokens implique mejor rendimiento downstream; la pre-tokenización pesa más de lo que se pensaba. Y Bostrom & Durrett (2020) encuentran que Unigram LM (módulo 03) iguala o supera a BPE al preentrenar. BPE es *estándar*, no *óptimo*.

## 5. De caracteres a **bytes** y la pre-tokenización con regex

### 5.1 Byte-level
Si el alfabeto inicial son los **256 bytes**, no hay desconocidos: cualquier texto (cualquier idioma, emoji, bytes inválidos) es una secuencia de bytes. El módulo 01 mostró lo que cuesta cada idioma en bytes; BPE reduce ese costo para lo frecuente.

**El truco de GPT-2 (`bytes_to_unicode`).** El código original de GPT-2 trabaja con *cadenas*, así que mapea cada byte a un carácter Unicode «imprimible» (los bytes imprimibles se mapean a sí mismos y el resto a `U+0100…`). Por eso verás `Ġ` (espacio) y `Ċ` (salto de línea) en los vocabularios de Hugging Face; es solo una *representación* de los bytes. Lo necesitaremos para HF `tokenizers`.

In [ ]:
def bytes_to_unicode():
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))
    cs, n = bs[:], 0
    for b in range(256):
        if b not in bs: bs.append(b); cs.append(256 + n); n += 1
    return dict(zip(bs, map(chr, cs)))
B2U = bytes_to_unicode()
a_str = lambda b: "".join(B2U[x] for x in b)
print("espacio →", repr(B2U[0x20]), "| salto de línea →", repr(B2U[0x0A]), "| 'a' →", repr(B2U[0x61]), "| byte 0xC3 →", repr(B2U[0xC3]), "| distintos:", len(set(B2U.values())))
print("'¿qué?' como bytes de GPT-2:", a_str("¿qué?".encode()))

### 5.2 Por qué hace falta un regex
Si dejas a BPE operar sobre todo el texto, aprende fusiones **entre** palabra y puntuación («dog», «dog.», «dog!», «dog,»… como tokens distintos) y entre palabras, y desperdicia vocabulario. GPT-2 (Radford et al., 2019, sec. 2.2) evita fusionar entre **categorías de caracteres** (letras, números, puntuación), con una excepción para el **espacio**, que se mantiene pegado a la palabra siguiente porque eso comprime mucho mejor. Se implementa **partiendo antes** el texto con una expresión regular; las fusiones solo ocurren *dentro* de cada trozo. Además, entrenar sobre trozos únicos con frecuencia (como en Sennrich) acelera todo.

### 5.3 Tres generaciones de regex (copiados de `tiktoken` 0.14.0)
* **GPT-2 (`r50k`)**: contracciones (`'s`, `'t`…), ` ?\p{L}+` (palabra con espacio previo), ` ?\p{N}+` (**números de cualquier longitud**), ` ?[^\s\p{L}\p{N}]+` (símbolos), y reglas para espacios.
* **cl100k (GPT-3.5/4)**: contracciones **sin distinguir mayúsculas**; la letra puede ir precedida de **un carácter no alfanumérico** (`[^\r\n\p{L}\p{N}]?+\p{L}++`, de ahí `_var` o `¡Hola`); **números en grupos de 1 a 3 dígitos** (`\p{N}{1,3}`); los símbolos absorben saltos de línea finales.
* **o200k (GPT-4o y posteriores)**: palabras con **clases de mayúsculas/minúsculas** (`camelCase` se parte en `camel` + `Case`), contracciones pegadas a la palabra, dígitos 1–3, y símbolos con `[\r\n/]*`.

In [ ]:
import regex

# Patrones de pre-tokenización copiados de tiktoken 0.14.0 (tiktoken_ext/openai_public.py; verificados el 25-oct-2026)
PAT_GPT2 = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}++| ?\p{N}++| ?[^\s\p{L}\p{N}]++|\s++$|\s+(?!\S)|\s"""
PAT_CL100K = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}++|\p{N}{1,3}+| ?[^\s\p{L}\p{N}]++[\r\n]*+|\s++$|\s*[\r\n]|\s+(?!\S)|\s"""
PAT_O200K = "|".join([
    r"""[^\r\n\p{L}\p{N}]?[\p{Lu}\p{Lt}\p{Lm}\p{Lo}\p{M}]*[\p{Ll}\p{Lm}\p{Lo}\p{M}]+(?i:'s|'t|'re|'ve|'m|'ll|'d)?""",
    r"""[^\r\n\p{L}\p{N}]?[\p{Lu}\p{Lt}\p{Lm}\p{Lo}\p{M}]+[\p{Ll}\p{Lm}\p{Lo}\p{M}]*(?i:'s|'t|'re|'ve|'m|'ll|'d)?""",
    r"""\p{N}{1,3}""", r""" ?[^\s\p{L}\p{N}]+[\r\n/]*""", r"""\s*[\r\n]+""", r"""\s+(?!\S)""", r"""\s+"""])
PAT_ESPACIOS = r"""\s?\S+|\s+"""      # línea base SIN categorías: corta solo por espacios (puntuación pegada a la palabra)
PATRONES = {"gpt2": PAT_GPT2, "cl100k": PAT_CL100K, "o200k": PAT_O200K, "solo espacios": PAT_ESPACIOS}

In [ ]:
import tiktoken, tiktoken_ext.openai_public as op
assert PAT_GPT2 == op.r50k_pat_str                      # el que trae tu versión de tiktoken
try:
    for nombre, pat in (("cl100k_base", PAT_CL100K), ("o200k_base", PAT_O200K)):
        assert tiktoken.get_encoding(nombre)._pat_str == pat, nombre
    print("✔ patrones cl100k y o200k idénticos a los de tiktoken instalado")
except Exception as e:
    print(f"(no pude descargar cl100k/o200k para contrastar: {type(e).__name__}; los patrones son los de tiktoken 0.14.0)")

ORIG = r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""   # GPT-2 original (sin cuantificadores posesivos)
texto = "\n".join(d["text"] for d in docs[:60]) + "  ¡Hola!  \n\n\tx=1  "
assert regex.findall(ORIG, texto) == regex.findall(PAT_GPT2, texto)
print("✔ el regex original de GPT-2 y r50k_pat_str trocean igual (tiktoken lo dice en un comentario: «equivalent, but faster»)")

In [ ]:
muestras = ["Don't pay $1234567 for camelCase_var, OK?", "¡Hola, MundoGrande! Año 2026: 3.14159", "def f(x):\n    return x  \n\n\tif y:"]
filas = []
for m in muestras:
    for nombre in ("gpt2", "cl100k", "o200k"):
        filas.append({"texto": m.replace("\n", "⏎").replace("\t", "⇥")[:34], "regex": nombre, "trozos": " | ".join(t.replace("\n", "⏎").replace("\t", "⇥") for t in regex.findall(PATRONES[nombre], m))})
pd.set_option("display.max_colwidth", 200); display(pd.DataFrame(filas).set_index(["texto", "regex"]))

> ### 🎨 GEMINI-IMAGEN · ID: `M02-IMG-02` · ⚠️ REEMPLAZAR ESTA CELDA
> **Cómo trocean GPT-2, cl100k y o200k la misma frase**
>
> **Qué muestra:** Una comparación visual de los trozos que producen los tres regex de pre-tokenización sobre dos frases, resaltando las diferencias clave: dígitos, camelCase, espacios y puntuación.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Tabla de trozos de tres regex de pre-tokenización sobre dos frases con las diferencias resaltadas
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, 16:9. Paleta: azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280. Tipografía monoespaciada para los trozos, sans-serif para títulos. Sin ilustraciones decorativas.
> Composición: una tabla con 3 filas (una por regex) y 2 bloques, uno por frase. Cada trozo es una cajita con borde fino; el carácter espacio se representa con el símbolo «␣» dentro de la cajita; el salto de línea con «⏎». Los textos son EXACTOS.
> Frase A (título del bloque): «Don't pay $1234567 for camelCase_var».
> Fila «GPT-2»: «Don» | «'t» | «␣pay» | «␣$» | «1234567» | «␣for» | «␣camelCase» | «_» | «var».
> Fila «cl100k»: «Don» | «'t» | «␣pay» | «␣$» | «123» | «456» | «7» | «␣for» | «␣camelCase» | «_var».
> Fila «o200k»: «Don't» | «␣pay» | «␣$» | «123» | «456» | «7» | «␣for» | «␣camel» | «Case» | «_var».
> Frase B (título del bloque): «¡Hola, MundoGrande! Año 2026».
> Fila «GPT-2»: «¡» | «Hola» | «,» | «␣MundoGrande» | «!» | «␣Año» | «␣2026».
> Fila «cl100k»: «¡Hola» | «,» | «␣MundoGrande» | «!» | «␣Año» | «␣» | «202» | «6».
> Fila «o200k»: «¡Hola» | «,» | «␣Mundo» | «Grande» | «!» | «␣Año» | «␣» | «202» | «6».
> Resalta en naranja las cajitas que difieren entre filas de un mismo bloque. Título superior: «Mismo texto, tres pre-tokenizadores». Nota inferior en gris: «cl100k y o200k parten los números en grupos de hasta 3 dígitos; o200k parte camelCase».
> Evita: cambiar los trozos, texto ilegible, más o menos cajitas de las indicadas, decoración.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M02-IMG-02.png` en la carpeta del módulo y reemplaza esta celda por: `![Tabla de trozos de tres regex de pre-tokenización sobre dos frases con las diferencias resaltadas](assets/M02-IMG-02.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> Don't pay $1234567 for camelCase_var
>  GPT-2 : Don|'t|␣pay|␣$|1234567|␣for|␣camelCase|_|var
>  cl100k: Don|'t|␣pay|␣$|123|456|7|␣for|␣camelCase|_var
>  o200k : Don't|␣pay|␣$|123|456|7|␣for|␣camel|Case|_var
> ¡Hola, MundoGrande! Año 2026
>  GPT-2 : ¡|Hola|,|␣MundoGrande|!|␣Año|␣2026
>  cl100k: ¡Hola|,|␣MundoGrande|!|␣Año|␣|202|6
>  o200k : ¡Hola|,|␣Mundo|Grande|!|␣Año|␣|202|6
> ```

> 👀 **Qué debes observar.** Tres diferencias con consecuencias reales: (1) los **números**: GPT-2 los deja enteros (`1234567` es un único trozo, a veces un único token) y cl100k/o200k los parten **de izquierda a derecha en grupos de 3**, de modo que la alineación de los dígitos depende de dónde empieza el número (Singh & Strouse, 2024, estudian el efecto en aritmética); (2) **camelCase** solo se parte en o200k; (3) el **espacio** de `␣2026` se separa del número en cl100k/o200k (`␣` + `202` + `6`).

## 6. Entrenamiento: del ingenuo al incremental

**Ingenuo** (Algoritmo 1 de Sennrich): en cada iteración recorre todo el corpus, recuenta todos los pares y fusiona. Coste $O(M\cdot U)$ con $U$ = nº de símbolos del corpus de palabras únicas. **Incremental** (lo que Sennrich et al. llaman «indexar todos los pares y actualizar las estructuras incrementalmente»): se mantienen **(1)** un contador de pares, **(2)** un **índice invertido** par → palabras donde aparece y **(3)** un **heap** con los conteos (con invalidación perezosa de entradas obsoletas). Fusionar un par solo toca las palabras de su lista, y por cada fusión actualiza los conteos de los pares **vecinos** (los que se destruyen y los que nacen).

> ### 🎨 GEMINI-IMAGEN · ID: `M02-IMG-03` · ⚠️ REEMPLAZAR ESTA CELDA
> **Estructuras del entrenamiento incremental de BPE**
>
> **Qué muestra:** Las tres estructuras (palabras, contador de pares con heap, índice invertido) y qué se actualiza al fusionar el par «e s»; sirve para entender el código de `entrena_incremental` antes de leerlo.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Tres columnas: palabras, contador de pares con heap e índice invertido antes y después de fusionar e+s
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, 16:9. Paleta: azul #1F6FEB para palabras, naranja #F28E2B para el contador/heap, verde #2CA02C para el índice invertido, gris #6B7280 para anotaciones. Tipografía monoespaciada para los datos, sans-serif para títulos. Sin ilustraciones decorativas.
> Composición: tres columnas con título arriba y una flecha etiquetada «fusionar (e, s)» que cruza de izquierda a derecha.
> Columna 1, título «Palabras (únicas, con frecuencia)»: dos filas EXACTAS «n e w e s t </w> ×6» y «w i d e s t </w> ×3». Debajo, la misma columna después de fusionar: «n e w es t </w> ×6» y «w i d es t </w> ×3» con «es» resaltado en verde.
> Columna 2, título «Contador de pares (+ heap)»: lista EXACTA antes: «(e,s): 9», «(s,t): 9», «(t,</w>): 9», «(w,e): 6», «(d,e): 3». Debajo, después de fusionar: «(e,s): 0 → se elimina», «(s,t): 0», «(es,t): 9 ← nuevo», «(w,es): 6 ← nuevo», «(d,es): 3 ← nuevo», «(w,e): 0», «(d,e): 0». Una nota: «solo cambian los pares vecinos del par fusionado».
> Columna 3, título «Índice invertido (par → palabras)»: antes «(e,s) → {newest, widest}»; después «(es,t) → {newest, widest}», «(w,es) → {newest}», «(d,es) → {widest}».
> Título superior: «Entrenamiento incremental de BPE». Nota inferior en gris: «cada merge toca solo las palabras de la lista del par».
> Evita: cambiar los números, añadir estructuras no indicadas, texto ilegible, decoración.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M02-IMG-03.png` en la carpeta del módulo y reemplaza esta celda por: `![Tres columnas: palabras, contador de pares con heap e índice invertido antes y después de fusionar e+s](assets/M02-IMG-03.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> PALABRAS            CONTADOR DE PARES            ÍNDICE INVERTIDO
> n e w e s t ×6      (e,s):9 (s,t):9 (t,</w>):9    (e,s)→{newest,widest}
> w i d e s t ×3      (w,e):6 (d,e):3
>    ── fusionar (e,s) ──────────────────────────────►
> n e w es t ×6       (es,t):9 nuevo (w,es):6 nuevo  (es,t)→{newest,widest}
> w i d es t ×3       (d,es):3 nuevo; (e,s),(w,e),(d,e)→0   (w,es)→{newest} (d,es)→{widest}
> ```

> 👀 **Qué debes observar.** Mira la columna central: al fusionar `(e,s)` **no** hay que recontar el corpus; basta restar los pares viejos que rodeaban a la fusión y sumar los nuevos. Cuando el mismo par vecino aparece en muchas palabras, el índice invertido evita tocar las que no lo contienen.

In [ ]:
import heapq
from collections import Counter, defaultdict


def contar_palabras(textos, pat_str=PAT_GPT2):
    """Corpus -> {palabra (tupla de bytes): frecuencia}. Entrenar sobre palabras únicas ponderadas es lo que hace Sennrich."""
    pat = regex.compile(pat_str)
    return Counter(tuple(w.encode("utf-8")) for t in textos for w in pat.findall(t))


def fusiona_palabra(w, par, nuevo):
    a, b = par; out, i = [], 0
    while i < len(w):
        if i < len(w) - 1 and w[i] == a and w[i + 1] == b:
            out.append(nuevo); i += 2
        else:
            out.append(w[i]); i += 1
    return out


def entrena_ingenuo(wc, n_merges, tie="lex"):
    """O(M · U): recuenta TODOS los pares en cada iteración. tie='lex' (menor par) o 'first' (primer visto, como tiktoken._educational)."""
    palabras, frec, merges = [list(w) for w in wc], list(wc.values()), []
    for k in range(n_merges):
        stats = Counter()
        for w, f in zip(palabras, frec):
            for p in zip(w, w[1:]):
                stats[p] += f
        if not stats:
            break
        par = min(stats, key=lambda p: (-stats[p], p)) if tie == "lex" else max(stats, key=stats.get)
        merges.append(par)
        palabras = [fusiona_palabra(w, par, 256 + k) for w in palabras]
    return merges

In [ ]:
def entrena_incremental(wc, n_merges):
    """Conteo INCREMENTAL: contadores de pares + índice invertido par -> palabras + heap con invalidación perezosa.
    Cada merge solo toca las palabras que contienen el par y ajusta los contadores de sus vecinos."""
    palabras, frec = [list(w) for w in wc], list(wc.values())
    stats, donde = Counter(), defaultdict(set)
    for i, (w, f) in enumerate(zip(palabras, frec)):
        for p in zip(w, w[1:]):
            stats[p] += f; donde[p].add(i)
    heap = [(-c, p) for p, c in stats.items()]; heapq.heapify(heap); merges = []
    while len(merges) < n_merges and heap:
        negc, par = heapq.heappop(heap)
        if stats.get(par, 0) != -negc:                       # entrada obsoleta: el contador cambió desde que se metió
            continue
        nuevo, (a, b), cambiados = 256 + len(merges), par, set(); merges.append(par)
        for i in list(donde[par]):
            w, f, out, j = palabras[i], frec[i], [], 0
            while j < len(w):
                if j < len(w) - 1 and w[j] == a and w[j + 1] == b:
                    if out:                                  # vecino izquierdo
                        stats[(out[-1], a)] -= f; stats[(out[-1], nuevo)] += f; donde[(out[-1], nuevo)].add(i)
                        cambiados |= {(out[-1], a), (out[-1], nuevo)}
                    if j + 2 < len(w):                       # vecino derecho
                        stats[(b, w[j + 2])] -= f; stats[(nuevo, w[j + 2])] += f; donde[(nuevo, w[j + 2])].add(i)
                        cambiados |= {(b, w[j + 2]), (nuevo, w[j + 2])}
                    stats[par] -= f; out.append(nuevo); j += 2
                else:
                    out.append(w[j]); j += 1
            palabras[i] = out
        del stats[par]; donde.pop(par, None)
        for p in cambiados - {par}:
            c = stats.get(p, 0)
            if c > 0:
                heapq.heappush(heap, (-c, p))
            else:
                stats.pop(p, None)
    return merges

**Convención de desempates.** Cuando dos pares tienen el mismo conteo hay que elegir uno, y **el vocabulario resultante cambia**. Aquí: mayor conteo y, ante empate, el **par menor** (`(id_a, id_b)` lexicográfico) → determinista y reproducible. `tiktoken._educational` y `minbpe` usan *«el primero que aparece»* (`max(stats, key=stats.get)`); HF `tokenizers` usa otra regla interna. Lo verificamos en la sección 8.

**Verificación 1: el incremental produce *exactamente* los mismos merges que el ingenuo** (misma regla de desempate), y es mucho más rápido.

In [ ]:
wc = contar_palabras(train["es"] + train["en"] + train["code"])
print(f"palabras únicas: {len(wc):,} | símbolos totales (bytes): {sum(len(w) * f for w, f in wc.items()):,}")
tiempos = []
for M in ((50, 100, 200, 400) if FAST_DEV_RUN else (100, 300, 600, 1000)):
    t0 = time.time(); m1 = entrena_ingenuo(wc, M); t1 = time.time() - t0
    t0 = time.time(); m2 = entrena_incremental(wc, M); t2 = time.time() - t0
    assert m1 == m2, "el incremental debe coincidir con el ingenuo"
    tiempos.append((M, t1, t2))
df_t = pd.DataFrame(tiempos, columns=["merges", "ingenuo (s)", "incremental (s)"]).set_index("merges"); df_t["aceleración ×"] = df_t["ingenuo (s)"] / df_t["incremental (s)"]
display(df_t.round(3))
ax = df_t[["ingenuo (s)", "incremental (s)"]].plot(logy=True, marker="o", color=[NARANJA, AZUL], figsize=(6.5, 3.6))
ax.set_ylabel("segundos (escala log)"); ax.set_title("Tiempo de entrenamiento vs nº de merges"); plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** Las dos curvas **dan los mismos merges** (la aserción lo comprueba) pero el ingenuo crece con $M\times U$ y el incremental casi no se nota. Con corpus reales de GB, el ingenuo es inviable; por eso existen implementaciones en Rust (`tokenizers`, `tiktoken`) y trabajos que mejoran la complejidad de la **codificación** de $O(NM)$ a $O(N\log M)$ (Zouhar et al., 2023).

## 7. El tokenizer: `encode`, `decode`, especiales y orden de merges
`BPETokenizer` guarda `vocab` (id → bytes) y `merges` ((id_a, id_b) → id nuevo). El **id nuevo es la prioridad**: al codificar, en cada paso se fusiona el par adyacente de **menor** id resultante. Los ids 0–255 son los bytes; los tokens especiales (como `<|endoftext|>`) viven **fuera** del BPE y se insertan con `allowed_special`.

In [ ]:
class BPETokenizer:
    """Byte-level BPE. vocab: id -> bytes. merges: (id_a, id_b) -> id nuevo; el id nuevo ES la prioridad (menor = antes)."""

    def __init__(self, pat_str=PAT_GPT2, vocab=None, merges=None, special_tokens=None):
        self.pat_str, self.pat = pat_str, regex.compile(pat_str)
        self.vocab = vocab or {i: bytes([i]) for i in range(256)}
        self.merges = merges or {}
        self.special = special_tokens or {}
        self.byte2id = {b: i for i, b in self.vocab.items() if len(b) == 1}
        self._cache = {}

    @classmethod
    def train(cls, textos, vocab_size, pat_str=PAT_GPT2, tie="lex"):
        wc = contar_palabras(textos, pat_str)
        lista = entrena_incremental(wc, vocab_size - 256) if tie == "lex" else entrena_ingenuo(wc, vocab_size - 256, tie)
        vocab, merges = {i: bytes([i]) for i in range(256)}, {}
        for k, (a, b) in enumerate(lista):
            vocab[256 + k] = vocab[a] + vocab[b]; merges[(a, b)] = 256 + k
        return cls(pat_str, vocab, merges)

    def _encode_trozo(self, trozo: str, max_id=None):
        clave = (trozo, max_id)
        if clave not in self._cache:
            ids = [self.byte2id[bytes([b])] for b in trozo.encode("utf-8")]
            while len(ids) > 1:
                cand = [(self.merges.get(p, 1 << 60), p) for p in zip(ids, ids[1:])]
                rango, par = min(cand)                       # el par de MENOR id = el que se aprendió antes
                if rango == 1 << 60 or (max_id is not None and rango >= max_id):
                    break
                ids = fusiona_palabra(ids, par, rango)
            self._cache[clave] = ids
        return self._cache[clave]

    def encode(self, texto: str, allowed_special=(), max_id=None) -> list[int]:
        partes = [texto]
        if allowed_special:
            partes = regex.split("(" + "|".join(map(regex.escape, allowed_special)) + ")", texto)
        ids = []
        for p in partes:
            if p in self.special and p in allowed_special:
                ids.append(self.special[p])
            else:
                ids += [i for t in self.pat.findall(p) for i in self._encode_trozo(t, max_id)]
        return ids

    def decode_bytes(self, ids) -> bytes:
        inv = {v: k.encode() for k, v in self.special.items()}
        return b"".join(self.vocab[i] if i in self.vocab else inv[i] for i in ids)

    def decode(self, ids) -> str:
        return self.decode_bytes(ids).decode("utf-8", errors="replace")

In [ ]:
tok = BPETokenizer.train(train["es"] + train["en"] + train["code"], VOCAB)
tok.special = {"<|endoftext|>": len(tok.vocab)}
print(f"vocabulario: {len(tok.vocab)} tokens (+{len(tok.special)} especial) | merges: {len(tok.merges)}")
mostrar = lambda ids: [tok.vocab[i].decode("utf-8", "backslashreplace") if i in tok.vocab else "<ESP>" for i in ids]
print("primeros 12 merges :", [tok.vocab[i] for i in range(256, 268)])
print("últimos 8 merges   :", [tok.vocab[i] for i in range(len(tok.vocab) - 8, len(tok.vocab))])
for s in ["La caché del servidor", "The cache of the server", "def configurar_cache(x):", "Año 2026 😀"]:
    ids = tok.encode(s); print(f"{s!r:34} → {len(ids):2d} tokens {mostrar(ids)}")
    assert tok.decode(ids) == s
ids = tok.encode("hola<|endoftext|>mundo", allowed_special={"<|endoftext|>"}); print(ids[-4:], repr(tok.decode(ids)))

In [ ]:
from IPython.display import HTML, display
def visualiza(texto, tk=None):
    tk = tk or tok; colores = ["#cfe2ff", "#ffe5c2", "#d4edda", "#e2d9f3"]; partes = []
    for k, i in enumerate(tk.encode(texto)):
        pieza = tk.vocab[i].decode("utf-8", "replace").replace(" ", "␣").replace("\n", "⏎")
        partes.append(f'<span style="background:{colores[k % 4]};padding:1px 2px;margin:1px;border-radius:3px;font-family:monospace">{pieza}</span>')
    display(HTML("".join(partes)))
for s in ("La configuración de la caché del servidor.", "The server cache configuration.", "def configure_cache(x): return x + 1", "Ñandú 😀"):
    visualiza(s)

> 👀 **Qué debes observar.** Compara el **español** con el **inglés**: con un vocabulario pequeño aprendido sobre el corpus mixto, las palabras frecuentes de cada idioma son un solo token y las raras se parten. En «Ñandú 😀» verás **piezas de un byte** (`\xc3`, `\x91`…) que no son caracteres válidos por sí solas: un token puede ser *medio carácter* UTF-8.

In [ ]:
# Un token puede terminar a mitad de un carácter: decodificar token a token produce basura; hay que decodificar los BYTES acumulados
ids = tok.encode("Ñandú 😀")
por_token = [tok.vocab[i].decode("utf-8", "replace") for i in ids]
print("decodificar cada token por separado:", por_token)
print("decodificar todos los bytes juntos :", tok.decode(ids))
import codecs
dec = codecs.getincrementaldecoder("utf-8")(); print("decodificador incremental        :", [dec.decode(tok.vocab[i]) for i in ids])

## 8. Verificación contra las librerías de industria
Una implementación «que se ve bien» no vale: la **verificamos** contra lo que usa la industria. Cuatro comprobaciones:

1. **tiktoken**: construimos un `tiktoken.Encoding` con *mis* merges (`mergeable_ranks = {bytes: id}`) y comparamos los ids de `encode` en una muestra variada (español, inglés, código, emojis, espacios raros).
2. **Hugging Face `tokenizers`**: cargamos mis merges en un modelo `BPE` con pre-tokenizer `Split(regex) + ByteLevel` y comparamos.
3. **Entrenamiento**: mi entrenador con desempate «primer visto» debe igualar a `tiktoken._educational.bpe_train`; y el entrenador de HF diverge en un desempate que podemos identificar.
4. **Vocabulario real de GPT-2** (50 257 tokens): recuperamos sus merges desde `tiktoken` y comprobamos que **mi** `encode` reproduce exactamente a `tiktoken` sobre texto multilingüe.

In [ ]:
import tiktoken


def a_tiktoken(tok, nombre="mi_bpe"):
    """Mi tokenizer como tiktoken.Encoding: mergeable_ranks = {bytes del token: id}. Los ids ya son las prioridades."""
    ranks = {tok.vocab[i]: i for i in sorted(tok.vocab)}
    assert len(ranks) == len(tok.vocab), "hay dos tokens con los mismos bytes: tiktoken no lo admite"
    return tiktoken.Encoding(nombre, pat_str=tok.pat_str, mergeable_ranks=ranks, special_tokens=dict(tok.special))


def bpe_bytes(ranks, pieza: bytes, max_rank=None):
    """Algoritmo de tiktoken sobre bytes: fusiona el par adyacente cuya unión tenga menor rango (< max_rank)."""
    partes = [bytes([b]) for b in pieza]
    while len(partes) > 1:
        mejor = min(((ranks.get(a + b), i) for i, (a, b) in enumerate(zip(partes, partes[1:])) if ranks.get(a + b) is not None), default=None)
        if mejor is None or (max_rank is not None and mejor[0] >= max_rank):
            break
        i = mejor[1]; partes[i:i + 2] = [partes[i] + partes[i + 1]]
    return partes


def desde_tiktoken(enc):
    """Recupera MIS estructuras (vocab, merges) desde un tiktoken.Encoding real (receta de minbpe: re-fusionar cada token con max_rank=su rango)."""
    ranks = enc._mergeable_ranks
    vocab = {r: b for b, r in ranks.items()}
    merges = {}
    for b, r in ranks.items():
        if len(b) > 1:
            a1, a2 = bpe_bytes(ranks, b, max_rank=r)
            merges[(ranks[a1], ranks[a2])] = r
    return BPETokenizer(enc._pat_str, vocab, merges, dict(enc._special_tokens))

In [ ]:
muestra = [d["text"] for d in docs[::13]][:25] + ["Hola, ¿cómo estás? 😀 你好\n\n  x", "naïve café 123456 ", "   ", "a\tb\r\nc  ", "don't DON'T", "x=3.14159;y=-42", ""]
enc_mio = a_tiktoken(tok)
n_ok = sum(tok.encode(t) == enc_mio.encode(t) for t in muestra)
print(f"tiktoken(mis merges) vs mi encode: {n_ok}/{len(muestra)} textos idénticos | decode ida y vuelta:", all(enc_mio.decode(enc_mio.encode(t)) == t for t in muestra))
assert n_ok == len(muestra)

# HF tokenizers con mis merges (ByteLevel usa el alfabeto bytes_to_unicode)
from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers, Regex
GPT2_ORIG = r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
def hf_desde_mio(tk, pat=GPT2_ORIG):
    vocab = {a_str(tk.vocab[i]): i for i in sorted(tk.vocab)}
    merges = [(a_str(tk.vocab[a]), a_str(tk.vocab[b])) for (a, b), _ in sorted(tk.merges.items(), key=lambda kv: kv[1])]
    hf = Tokenizer(models.BPE(vocab=vocab, merges=merges))
    hf.pre_tokenizer = pre_tokenizers.Sequence([pre_tokenizers.Split(Regex(pat), behavior="isolated"), pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=False)])
    hf.decoder = decoders.ByteLevel(); return hf
hf = hf_desde_mio(tok)
n_ok = sum(tok.encode(t) == hf.encode(t).ids for t in muestra)
print(f"HF tokenizers(mis merges) vs mi encode: {n_ok}/{len(muestra)} idénticos | decode:", all(hf.decode(hf.encode(t).ids) == t for t in muestra))
assert n_ok == len(muestra)

In [ ]:
# Entrenamiento: (a) tiktoken._educational  (b) HF BpeTrainer
texto_corto = "\n".join(train["es"] + train["en"] + train["code"])[:50_000]; V = 360
wc_c = contar_palabras([texto_corto])
mio_first = entrena_ingenuo(wc_c, V - 256, tie="first")
try:
    from tiktoken._educational import bpe_train
    ranks_edu = bpe_train(data=texto_corto, vocab_size=V, pat_str=PAT_GPT2, visualise=None)
    bytes_edu = [b for b, r in sorted(ranks_edu.items(), key=lambda kv: kv[1])][256:]
    vocab_b = {i: bytes([i]) for i in range(256)}
    for k, (a, b) in enumerate(mio_first): vocab_b[256 + k] = vocab_b[a] + vocab_b[b]
    print("mi entrenador (desempate «primer visto») == tiktoken._educational:", [vocab_b[256 + k] for k in range(len(mio_first))] == bytes_edu)
except Exception as e:
    print("tiktoken._educational no disponible:", type(e).__name__)

mio_lex = BPETokenizer.train([texto_corto], V)
hf_tr = Tokenizer(models.BPE()); hf_tr.pre_tokenizer = pre_tokenizers.Sequence([pre_tokenizers.Split(Regex(GPT2_ORIG), behavior="isolated"), pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=False)])
hf_tr.train_from_iterator([texto_corto], trainers.BpeTrainer(vocab_size=V, initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), special_tokens=[], show_progress=False))
import json
m_hf = [tuple(m) if isinstance(m, list) else tuple(m.split(" ")) for m in json.loads(hf_tr.to_str())["model"]["merges"]]
m_mio = [(a_str(mio_lex.vocab[a]), a_str(mio_lex.vocab[b])) for (a, b), _ in sorted(mio_lex.merges.items(), key=lambda kv: kv[1])]
k = next((i for i, (x, y) in enumerate(zip(m_hf, m_mio)) if x != y), None)
print(f"HF BpeTrainer vs mi entrenador (desempate lexicográfico): coinciden los primeros {k} de {len(m_mio)} merges" if k is not None else "idénticos")
if k is not None:
    print("  HF  elige :", m_hf[k], "| yo elijo:", m_mio[k], "(¿mismo conteo? ver siguiente celda)")

In [ ]:
# ¿Es un desempate? Cuenta los dos pares candidatos sobre el corpus tras aplicar los k merges comunes
if k is not None:
    U2B = {v: k_ for k_, v in B2U.items()}; a_bytes = lambda s: bytes(U2B[c] for c in s)
    tok_k = BPETokenizer(PAT_GPT2, mio_lex.vocab, mio_lex.merges)
    def cuenta(par_hf):
        ca, cb = a_bytes(par_hf[0]), a_bytes(par_hf[1]); n = 0
        for w, f in wc_c.items():
            ids = [tok_k.byte2id[bytes([b])] for b in w]
            while len(ids) > 1:
                cand = [(tok_k.merges.get(p, 1 << 60), p) for p in zip(ids, ids[1:])]; r, p = min(cand)
                if r == 1 << 60 or r >= 256 + k: break
                ids = fusiona_palabra(ids, p, r)
            n += f * sum(1 for x, y in zip(ids, ids[1:]) if tok_k.vocab[x] == ca and tok_k.vocab[y] == cb)
        return n
    print("conteo del par que elige HF:", cuenta(m_hf[k]), "| conteo del par que elijo yo:", cuenta(m_mio[k]))

> 👀 **Qué debes observar.** 1. Con **mis merges**, `tiktoken` y `tokenizers` producen **exactamente** los mismos ids que mi `encode`: la parte de *codificación* (regex + orden de merges + bytes) está bien implementada.
> 2. Mi entrenador, con el desempate «primer visto», **reproduce a `tiktoken._educational`** token a token.
> 3. El entrenador de HF **diverge** de mi desempate lexicográfico en un punto en el que los dos candidatos tienen **el mismo conteo** (la celda de arriba lo cuenta): no hay un «BPE correcto» único; **la regla de desempate forma parte de la definición del tokenizer**. Lo que importa en producción es que el tokenizer entrenado se guarda y se aplica siempre igual.

### 8.4 Contra el vocabulario real de GPT-2
Obtenemos el encoding `gpt2` de `tiktoken` (o el espejo verificado por SHA-256 si el CDN está bloqueado), **recuperamos sus merges** con la receta de `minbpe` (volver a fusionar cada token con `max_rank` = su rango: el resultado son siempre exactamente dos piezas) y cargamos todo en **mi** `BPETokenizer`. Los ids deben coincidir con los de `tiktoken` (los ids de GPT-2 son sus rangos).

In [ ]:
import hashlib
import os
import tempfile
import urllib.request


def get_gpt2_encoding():
    """tiktoken 'gpt2'. 1) oficial (openaipublic); 2) espejo en GitHub (graykode/gpt-2-Pytorch) verificado con los
    mismos SHA-256 que exige tiktoken; 3) None (las celdas que lo usan se omiten con un aviso)."""
    import tiktoken
    try:
        return tiktoken.get_encoding("gpt2")
    except Exception as e:  # noqa: BLE001
        print(f"[tiktoken] descarga oficial no disponible ({type(e).__name__}); pruebo el espejo de GitHub...")
    try:
        import tiktoken_ext.openai_public as op
        from tiktoken.load import data_gym_to_mergeable_bpe_ranks
        base = "https://raw.githubusercontent.com/graykode/gpt-2-Pytorch/master/GPT2/"
        sha = {"vocab.bpe": "1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5",
               "encoder.json": "196139668be63f3b5d6574427317ae82f612a97c5d1cdaf36ed2256dbf636783"}
        paths = {}
        for name, h in sha.items():
            data = urllib.request.urlopen(base + name, timeout=60).read()
            assert hashlib.sha256(data).hexdigest() == h, f"hash de {name} no coincide"
            paths[name] = os.path.join(tempfile.gettempdir(), "gpt2_" + name)
            with open(paths[name], "wb") as f:
                f.write(data)
        ranks = data_gym_to_mergeable_bpe_ranks(vocab_bpe_file=paths["vocab.bpe"], encoder_json_file=paths["encoder.json"])
        print("[tiktoken] GPT-2 cargado desde el espejo de GitHub (hashes verificados)")
        return tiktoken.Encoding("gpt2", pat_str=op.r50k_pat_str, mergeable_ranks=ranks,
                                 special_tokens={"<|endoftext|>": 50256})
    except Exception as e:  # noqa: BLE001
        print(f"[tiktoken] sin GPT-2 ({type(e).__name__}: {str(e)[:80]}). Las comparaciones con tiktoken se omiten.")
        return None

In [ ]:
gpt2 = get_gpt2_encoding()
if gpt2 is None:
    print("GPT-2 no disponible: se omite esta verificación")
else:
    t0 = time.time(); mio_gpt2 = desde_tiktoken(gpt2); print(f"merges de GPT-2 recuperados: {len(mio_gpt2.merges):,} en {time.time() - t0:.1f} s | n_vocab = {gpt2.n_vocab:,} (= 256 + {len(mio_gpt2.merges):,} merges + 1 especial)")
    textos = ["Hello, world!", "Tokenizers turn text into numbers.", "Los tokenizadores convierten texto en números.", "def añadir_ñu(x): return x+1", "Año 2026: 😀 你好 мир",
              "   espacios   y\n\nsaltos\t\ttabs  ", "supercalifragilisticexpialidocious", "1234567890 3.14159 -42"] + [d["text"][:400] for d in docs[::40]]
    ok = [mio_gpt2.encode(t) == gpt2.encode(t) for t in textos]
    print(f"mi encode vs tiktoken en GPT-2 real: {sum(ok)}/{len(ok)} textos idénticos")
    print("Hello, world! →", mio_gpt2.encode("Hello, world!"), "(el de la figura M02-IMG-01: [15496, 11, 995, 0])")
    assert all(ok) and mio_gpt2.encode("Hello, world!") == [15496, 11, 995, 0]

In [ ]:
# Velocidad: mi Python vs HF tokenizers (Rust) vs tiktoken (Rust) sobre el mismo texto
grande = "\n".join(docs_t["text"] for docs_t in docs)[:300_000]
res_v = {}
for nombre, f in (("mi BPE (Python)", lambda: tok.encode(grande)), ("HF tokenizers", lambda: hf.encode(grande).ids), ("tiktoken", lambda: enc_mio.encode(grande))):
    t0 = time.time(); ids = f(); res_v[nombre] = len(grande.encode()) / (time.time() - t0) / 1e6
fig, ax = plt.subplots(figsize=(6.2, 3.3)); ax.barh(list(res_v), list(res_v.values()), color=[GRIS, NARANJA, AZUL]); ax.set_xscale("log")
ax.set_xlabel("MB/s de texto (escala log; una CPU compartida, un solo hilo)"); ax.set_title("Codificar con el mismo vocabulario"); plt.tight_layout(); plt.show()
print({k: f"{v:.2f} MB/s" for k, v in res_v.items()})

> 👀 **Qué debes observar.** Mismo vocabulario, mismas salidas, **velocidades distintas**. En esta medición (CPU compartida, un hilo, texto sintético muy repetitivo) `tiktoken` es el más rápido con diferencia; mi Python puro se ayuda de su caché de trozos repetidos; y la llamada `encode` de HF `tokenizers` **no** es la más rápida porque construye objetos con offsets y máscaras (se usa `encode_batch`, en paralelo, en producción). No saques conclusiones finas de este número: depende de la máquina, del texto y de la API usada. El README de `tiktoken` afirma 3–6× frente a un tokenizer de código abierto comparable (GPT-2 de HF con `tokenizers==0.13.2`), medido sobre 1 GB de texto.

## 9. Experimentos
### 9.1 Tamaño del vocabulario: un entrenamiento, todos los vocabularios
Propiedad útil: el vocabulario de tamaño $V'<V$ de un BPE es **el prefijo** de sus merges. Entrenamos una vez y evaluamos `encode(max_id=…)` con los primeros $k$ merges. Métrica: **bytes por token** en el conjunto de prueba (más alto = mejor compresión; el inverso de la «fertilidad por byte»).

In [ ]:
def trozos_prueba(textos, pat=PAT_GPT2):
    pat = regex.compile(pat); return Counter(t for x in textos for t in pat.findall(x))

def bytes_por_token(tk, trozos, max_id=None):
    nb_, nt = 0, 0
    for t, f in trozos.items():
        nb_ += f * len(t.encode("utf-8")); nt += f * len(tk._encode_trozo(t, max_id))
    return nb_ / nt

tp = {l: trozos_prueba(test[l]) for l in test}
Vs = sorted({256, 300, 400, 600, 800, VOCAB})
df_v = pd.DataFrame({l: [bytes_por_token(tok, tp[l], max_id=v) for v in Vs] for l in test}, index=Vs)
ax = df_v.plot(marker="o", color=[AZUL, NARANJA, VERDE], figsize=(7, 3.8)); ax.set_xlabel("tamaño del vocabulario"); ax.set_ylabel("bytes por token (prueba)")
ax.set_title(f"Compresión vs vocabulario (un solo entrenamiento, corpus {source})"); plt.tight_layout(); plt.show(); display(df_v.round(2))

> 👀 **Qué debes observar.** La curva **sube rápido y se aplana**: las primeras fusiones (pares muy frecuentes) comprimen mucho; las últimas apenas. Con $V=256$ (solo bytes) el ratio es exactamente 1.0 bytes/token. Elegir el tamaño de vocabulario es leer dónde la curva deja de compensar el costo de más filas en la matriz de embeddings (módulo 04).

### 9.2 Distribución de los tokens (ley de Zipf)

In [ ]:
usos = Counter(); idx_total = 0
for t, f in tp["es"].items():
    for i in tok._encode_trozo(t): usos[i] += f
for t, f in tp["en"].items():
    for i in tok._encode_trozo(t): usos[i] += f
frec = np.array(sorted(usos.values(), reverse=True)); n_vocab = len(tok.vocab)
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
axs[0].loglog(np.arange(1, len(frec) + 1), frec, color=AZUL); axs[0].set_xlabel("rango del token"); axs[0].set_ylabel("apariciones (ES+EN, prueba)"); axs[0].set_title("Frecuencia de tokens (log-log)")
acum = np.cumsum(frec) / frec.sum(); axs[1].plot(np.arange(1, len(frec) + 1), acum, color=NARANJA); axs[1].set_xlabel("nº de tokens más frecuentes"); axs[1].set_ylabel("fracción acumulada del texto"); axs[1].set_title("Cobertura acumulada")
plt.tight_layout(); plt.show()
print(f"tokens del vocabulario que NUNCA aparecen en la prueba ES+EN: {n_vocab - len(usos)} de {n_vocab} ({100 * (n_vocab - len(usos)) / n_vocab:.1f} %) | los 100 más frecuentes cubren {100 * acum[99]:.1f} % del texto")

> 👀 **Qué debes observar.** La frecuencia sigue (aproximadamente) una **ley de potencias**: unos pocos tokens cubren casi todo el texto y una **cola larga** casi no se usa. Los tokens con muy pocas apariciones tienen embeddings mal entrenados; con vocabularios enormes y datos distintos del entrenamiento del tokenizer aparecen los *tokens sub-entrenados* que verás en el módulo 05.

### 9.3 El «impuesto» de idioma: entrenar con un idioma y evaluar en otro
Entrenamos cuatro tokenizers con el **mismo** tamaño de vocabulario —solo inglés, solo español, solo código y la mezcla— y medimos los bytes por token en cada conjunto de prueba.

In [ ]:
conjuntos = {"solo EN": train["en"], "solo ES": train["es"], "solo código": train["code"], "mezcla": train["es"] + train["en"] + train["code"]}
tks = {n: BPETokenizer.train(ts, VOCAB) for n, ts in conjuntos.items()}
M = pd.DataFrame({l: {n: bytes_por_token(tk_, tp[l]) for n, tk_ in tks.items()} for l in ("en", "es", "code")})
M.columns = ["prueba EN", "prueba ES", "prueba código"]
import seaborn as sns
fig, ax = plt.subplots(figsize=(6.4, 3.6)); sns.heatmap(M, annot=True, fmt=".2f", cmap="Blues", cbar_kws={"label": "bytes por token"}, ax=ax)
ax.set_title(f"Tokenizer entrenado en… (filas) y evaluado en… (columnas)"); ax.set_ylabel("")
plt.tight_layout(); plt.show()
print("coste relativo de usar el tokenizer solo-EN en español:", round(M.loc["solo ES", "prueba ES"] / M.loc["solo EN", "prueba ES"], 2), "× más tokens")

> 👀 **Qué debes observar.** La **diagonal** es lo mejor de cada columna: cada tokenizer comprime bien *su* idioma. Fuera de la diagonal, el mismo texto cuesta **mucho más tokens** (la última línea lo cuantifica): ese es el «impuesto de tokens» que documentan Petrov et al. (2023) y Ahia et al. (2023) en tokenizers comerciales, y la razón de que Llama 3 añadiera 28 K tokens de idiomas no ingleses a los 100 K de `tiktoken`. La **mezcla** pierde algo en cada idioma respecto al especialista pero es el compromiso razonable. (Con el corpus sintético la diferencia se exagera porque cada idioma tiene un vocabulario muy limitado.)

### 9.4 El regex importa: ablación de la pre-tokenización

In [ ]:
def clase(c): return "L" if c.isalpha() else "N" if c.isdigit() else "P"
def mezcla(b):                       # ¿el token mezcla letras, números y/o símbolos (ignorando un espacio inicial)?
    s = b.decode("utf-8", "ignore").lstrip(" "); return len({clase(c) for c in s if not c.isspace()}) > 1

todo = train["es"] + train["en"] + train["code"]; filas = []
for nombre, pat in PATRONES.items():
    tk_ = BPETokenizer.train(todo, VOCAB, pat_str=pat); tpp = {l: trozos_prueba(test[l], pat) for l in test}
    filas.append({"pre-tokenizador": nombre, "bytes/token (ES)": bytes_por_token(tk_, tpp["es"]), "bytes/token (EN)": bytes_por_token(tk_, tpp["en"]),
                  "bytes/token (código)": bytes_por_token(tk_, tpp["code"]), "% tokens que mezclan letra/número/símbolo": 100 * np.mean([mezcla(b) for i, b in tk_.vocab.items() if i >= 256])})
df_r = pd.DataFrame(filas).set_index("pre-tokenizador"); display(df_r.round(2))
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
df_r[["bytes/token (ES)", "bytes/token (EN)", "bytes/token (código)"]].plot.bar(ax=axs[0], color=[AZUL, NARANJA, VERDE], rot=0); axs[0].set_title("Compresión según el pre-tokenizador"); axs[0].set_ylabel("bytes por token")
df_r["% tokens que mezclan letra/número/símbolo"].plot.bar(ax=axs[1], color=GRIS, rot=0); axs[1].set_title("Tokens que cruzan categorías"); axs[1].set_ylabel("% del vocabulario aprendido")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** 1. La línea base «**solo espacios**» deja la puntuación pegada a las palabras y aprende **bastante más tokens que mezclan letras y símbolos** (gráfica derecha): vocabulario gastado en variantes como `cache.`, `cache,`. Los regex de GPT-2/cl100k/o200k evitan casi toda esa mezcla (cl100k/o200k conservan los tokens con un símbolo inicial como `_var`).
> 2. En **compresión pura** (gráfica izquierda) «solo espacios» *no pierde*, incluso gana algo aquí: el regex **no** se introdujo para comprimir más, sino para que el vocabulario sea más limpio y generalice (GPT-2 solo hace una excepción con el espacio porque eso sí comprime). Lo mismo vale para los dígitos y el camelCase: son decisiones de **calidad del modelo**, no de compresión. Con Wikipedia real las diferencias entre generaciones de regex son más visibles en números y mayúsculas.

### 9.5 BPE como compresor: bits por byte
Si cada token se almacena con un código de **longitud fija** de $\lceil\log_2 |V|\rceil$ bits, el texto ocupa $n_{\text{tokens}}\cdot\log_2|V|$ bits. Un código de **Huffman** sobre la frecuencia de los tokens baja hacia la **entropía** $H_{\text{tok}}$ (cota inferior para un código de prefijo por símbolo): $n_{\text{tokens}}\cdot H_{\text{tok}}$ bits. Comparamos con compresores generales (`zlib` = LZ77 + Huffman, `bz2`, `lzma`) sobre los mismos bytes. *No incluimos el costo de almacenar el vocabulario/diccionario* (tampoco el de `zlib`, que no lo necesita).

In [ ]:
def bits_por_byte(texto_lista, tk, max_id=None):
    nb_ = nbytes(texto_lista); ids = [i for t in texto_lista for i in tk.encode(t, max_id=max_id)]
    cnt = Counter(ids); n = len(ids); H = -sum(v / n * math.log2(v / n) for v in cnt.values())
    return {"BPE longitud fija": n * math.ceil(math.log2(len(tk.vocab) if max_id is None else max_id)) / nb_, "BPE + Huffman (cota entropía)": n * H / nb_}

filas = []
for l in ("es", "en", "code"):
    crudo = "\n".join(test[l]).encode("utf-8"); n_ = len(crudo)
    fila = {"corpus": l, "sin comprimir": 8.0, **bits_por_byte(test[l], tok),
            "zlib": len(zlib.compress(crudo, 9)) * 8 / n_, "bz2": len(bz2.compress(crudo)) * 8 / n_, "lzma": len(lzma.compress(crudo)) * 8 / n_}
    filas.append(fila)
df_b = pd.DataFrame(filas).set_index("corpus"); display(df_b.round(2))
ax = df_b.plot.bar(figsize=(9, 3.8), color=[ROJO, AZUL, NARANJA, GRIS, "#8c564b", VERDE], rot=0, width=0.85); ax.set_ylabel("bits por byte de texto"); ax.set_title(f"BPE como compresor (vocab {VOCAB}) vs compresores generales")
ax.legend(fontsize=7, ncol=3); plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** 1. Un BPE con código de longitud fija **ya baja de 8 bits/byte**, y con codificación entrópica (Huffman) baja más: es compresión *orden 0 sobre tokens* (Shannon, 1948: no se puede bajar de la entropía por símbolo con un código de prefijo por símbolo).
> 2. Los compresores con **contexto** (LZ77 en `zlib`, BWT en `bz2`, `lzma`) ganan con claridad: BPE solo captura dependencias *locales* dentro de cada trozo. Esa dependencia a larga distancia es justo la que aprende el modelo de lenguaje: la métrica independiente del tokenizer con la que se comparan LLMs con vocabularios distintos es **bits por byte** del modelo (menos es mejor).
> 3. En el corpus sintético los compresores se disparan por la repetitividad del generador; con texto real su ventaja es menor, pero el orden relativo se mantiene.

### 9.6 El orden de los merges importa

In [ ]:
import random as _r
def seg_rango(tk, t): return tk._encode_trozo(t)
def seg_inverso(tk, t):                      # aplica el par de MAYOR id disponible (orden de merges invertido)
    ids = [tk.byte2id[bytes([b])] for b in t.encode()]
    while len(ids) > 1:
        c = [(tk.merges[p], p) for p in zip(ids, ids[1:]) if p in tk.merges]
        if not c: break
        r, p = max(c); ids = fusiona_palabra(ids, p, r)
    return ids
def seg_codicioso(tk, t):                    # prefijo más largo presente en el vocabulario (sin rangos), estilo MaxMatch
    b = t.encode(); inv = {v: k for k, v in tk.vocab.items()}; out, i = [], 0
    while i < len(b):
        L = next(L for L in range(min(len(b) - i, 16), 0, -1) if b[i:i + L] in inv); out.append(inv[b[i:i + L]]); i += L
    return out
def seg_azar(tk, t, rng=_r.Random(SEED)):
    ids = [tk.byte2id[bytes([b])] for b in t.encode()]
    while len(ids) > 1:
        c = [(tk.merges[p], p) for p in zip(ids, ids[1:]) if p in tk.merges]
        if not c: break
        r, p = rng.choice(c); ids = fusiona_palabra(ids, p, r)
    return ids
filas = []
for nombre, f in (("por rango (BPE)", seg_rango), ("rango invertido", seg_inverso), ("prefijo más largo", seg_codicioso), ("par al azar", seg_azar)):
    igual = tot = n_tok = n_byt = 0
    for t, fr in tp["es"].items():
        s = f(tok, t); igual += fr * (s == seg_rango(tok, t)); tot += fr; n_tok += fr * len(s); n_byt += fr * len(t.encode())
    filas.append({"orden de aplicación": nombre, "% trozos idénticos al de BPE": 100 * igual / tot, "bytes/token (ES)": n_byt / n_tok})
df_o = pd.DataFrame(filas).set_index("orden de aplicación"); display(df_o.round(2))
ax = df_o["% trozos idénticos al de BPE"].plot.bar(color=[AZUL, NARANJA, VERDE, GRIS], figsize=(6.5, 3.3), rot=0); ax.set_ylabel("% de trozos con la misma segmentación"); ax.set_title("Mismos merges, distinto orden de aplicación")
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** Con **el mismo conjunto de merges**, aplicarlos en otro orden (o con otra estrategia, como el prefijo más largo de WordPiece, módulo 03) cambia la segmentación de una fracción de los trozos y la compresión. Un modelo preentrenado con la segmentación por rango **no sabe leer** la otra: el *algoritmo de codificación* es parte del contrato del tokenizer, igual que el vocabulario.

## 10. Complejidad
| Operación | Ingenuo | Eficiente | Notas |
|---|---|---|---|
| **Entrenar** ($M$ merges, $U$ símbolos en palabras únicas) | $O(M\cdot U)$ | ≈ $O(U + \sum_k \text{occ}_k\log P)$ con heap ($P$ = nº de pares): cada merge toca solo las ocurrencias del par | Lo medimos en la sección 6 |
| **Codificar** un trozo de $n$ bytes | $O(n\cdot m)$ ($m$ = merges aplicables) | $O(n\log n)$ con cola de prioridad y lista enlazada | Zouhar et al. (2023): de $O(NM)$ a $O(N\log M)$ |
| **Decodificar** | $O(n)$ | $O(n)$ | concatenar bytes |
| **Memoria** | $O(\lvert V\rvert)$ | igual | + caché de trozos frecuentes |

La **caché de trozos** (`_cache`) acelera la codificación de texto natural, porque las palabras se repiten (Zipf). `tiktoken` además fusiona **un par por vez** (el de menor rango, el más a la izquierda en empates) y `HF tokenizers` mantiene una caché por palabra: el resultado es el mismo que el de aplicar todos los pares de una pasada, como verificamos.

## 11. Mapa de madurez
Estado a **octubre de 2026**.


| Técnica | Madurez (oct-2026) | Quién la usa (respaldado) | Fuente |
|---|---|---|---|
| Byte-level BPE + regex de pre-tokenización (familia GPT) | 🟢 | Encodings `gpt2`, `cl100k_base`, `o200k_base` de `tiktoken`; los modelos de OpenAI se asocian a ellos en `tiktoken/model.py` (p. ej. `gpt-4`→cl100k, `gpt-4o`→o200k, `gpt-5`→o200k en tiktoken 0.14.0) | Radford et al. 2019; repositorio `openai/tiktoken` |
| Vocabulario ampliado sobre un BPE existente | 🟢 | Llama 3: 100 K de `tiktoken` + 28 K tokens adicionales (128 K en total) | Llama Team 2024 (arXiv:2407.21783) |
| BPE (Sennrich) con fronteras de palabra | 🟢 | Traducción automática neuronal; base de muchos tokenizers | Sennrich et al. 2016 (arXiv:1508.07909) |
| Regularización: BPE-dropout | 🟡 | Resultados en traducción; adopción parcial | Provilkov et al. 2019 (arXiv:1910.13267) |
| Tokenizers que cruzan fronteras de palabra (SuperBPE) | 🟡 | Preentrenamientos de 8B parámetros reportados en el artículo (2025) | Liu et al. 2025 (arXiv:2503.13423) |
| Modelos sin tokenizer (BLT) | 🔴 | Investigación de Meta FAIR (módulo 06) | Pagnoni et al. 2024 (arXiv:2412.09871) |

## 12. 🏭 En producción
* **Casi todo LLM generalista de la familia GPT usa byte-level BPE.** En `tiktoken` 0.14.0, `gpt-4`/`gpt-3.5-turbo` → `cl100k_base`; `gpt-4o`, `gpt-4.1`, `o1`, `o3` y `gpt-5` → `o200k_base`; `gpt-oss` → `o200k_harmony` (tamaños de vocabulario: 100 277 y 200 019 contando especiales, según las constantes del código). Llama 3 parte de ese mismo tokenizer y lo amplía a 128 K.
* **Costo = tokens.** Cada token se factura y ocupa contexto: medir *tokens por carácter* de **tu** tráfico por idioma es una tarea de ingeniería habitual (módulo 04).
* **Se entrena una vez y se congela.** El tokenizer se entrena con una muestra representativa de **los datos del modelo** (no del dominio de la tarea), se serializa (vocabulario + merges + regex) y se usa idéntico en entrenamiento e inferencia; cambiarlo invalida el modelo (hay técnicas de *tokenizer transfer*, módulo 07).
* **Implementación rápida.** `tiktoken` y `tokenizers` están en Rust; Python puro sirve para entender, no para servir.
* **Tokens especiales y plantillas de chat** quedan fuera del BPE y se tratan aparte (`allowed_special`): módulo 04.

## 13. 🧠 Secretos de la élite
1. **El orden de los merges y los desempates son parte del tokenizer** (sección 9.6 y 8): dos BPE con las mismas fusiones en otro orden son incompatibles; HF y `tiktoken` desempatan distinto y por eso un vocabulario entrenado con uno no se «reproduce» con el otro aunque el algoritmo sea el mismo.
2. **Los dígitos de cl100k/o200k se parten de izquierda a derecha en grupos de 3**: `1234567` → `123|456|7`. Singh & Strouse (2024) muestran que forzar la partición de derecha a izquierda (separando con comas) **mejora la aritmética** de GPT-3.5/4. Un detalle de regex que condiciona una capacidad.
3. **El regex es más importante de lo que parece**: Schmidt et al. (2024) concluyen que la pre-tokenización pesa mucho en la calidad del tokenizer y que inicializar el vocabulario con BPE ayuda; comprimir más no implica un modelo mejor.
4. **Se entrena sobre palabras únicas ponderadas, no sobre el texto bruto** (Sennrich 2016): por eso el coste depende de la *diversidad* del corpus y no de su tamaño. En esta lección: ≈ 3 000 palabras únicas para ≈ 300 KB.
5. **Un token puede ser medio carácter UTF-8**: en *streaming* hay que decodificar bytes acumulados (sección 7). El `�` en salidas parciales de un LLM suele ser esto.
6. **Un tokenizer entrenado sobre datos distintos a los del modelo crea tokens que el modelo nunca vio** (tokens sub-entrenados): Land & Bartolo (2024) los detectan automáticamente; lo desarrollas en el módulo 05.
7. **La compresión del tokenizer no es comparable entre vocabularios** salvo normalizando por bytes: usa *bytes por token* o *bits por byte*, no «perplejidad por token».
8. **Verifica siempre contra una implementación de referencia** con *tus* merges cargados en ella (sección 8): es la forma barata de encontrar errores sutiles de regex, de orden o de bytes.

## 14. ⚠️ Errores comunes
* **Entrenar el tokenizer sobre texto sin normalizar** (o normalizar distinto en inferencia): módulo 01.
* **Aplicar los merges en un orden distinto** (p. ej. por frecuencia del par en el texto a codificar) y creer que «es lo mismo».
* **Olvidar el espacio inicial**: en GPT-2 `hello` y ` hello` son tokens distintos. Concatenar prompts sin cuidar los espacios cambia la tokenización (módulo 05: *token healing*).
* **Medir compresión sobre el mismo texto con el que entrenaste**: usa un conjunto de prueba.
* **Comparar vocabularios de distinto tamaño con «tokens por palabra»** sin fijar el corpus de evaluación.
* **Decodificar token a token y pegar cadenas**: parte caracteres multibyte.
* **Suponer que `tokenizers` y `tiktoken` entrenan igual**: coinciden al *codificar*, no al *entrenar* (desempates).

## 15. Autoevaluación

**1.** **(Cálculo)** Corpus: «aab»×3, «ab»×2 (sobre el alfabeto `a`, `b`, sin marcador de fin). Entrena BPE un paso con la regla «mayor conteo; empate → par menor». ¿Qué par se fusiona y cómo queda el corpus?

<details>
<summary>Ver respuesta</summary>

Pares: `(a,a)` aparece 1 vez en «aab» → 3 veces en total; `(a,b)`: 1 vez en «aab» (×3 = 3) + 1 en «ab» (×2 = 2) = 5. Gana `(a,b)` con 5 → token `ab`. Corpus: «a ab»×3 y «ab»×2 (es decir, `a`,`ab` y `ab`).

</details>

**2.** **(Cálculo)** Con un vocabulario de 50 000 merges + 256 bytes + 1 token especial, ¿cuál es `n_vocab` y cuántos bits necesitas con código de longitud fija por token?

<details>
<summary>Ver respuesta</summary>

`n_vocab` = 50 257 (es el de GPT-2). $\lceil\log_2 50257\rceil = 16$ bits por token.

</details>

**3.** **(Diagnóstico)** Entrenaste tu BPE con `tiktoken._educational` y lo cargas en HF `tokenizers` con el mismo regex, pero algunos textos tokenizan distinto. Da tres causas posibles.

<details>
<summary>Ver respuesta</summary>

(1) El regex difiere (posesivos `++` o clases Unicode entre motores); (2) falta el mapeo `bytes_to_unicode` o el pre-tokenizer `ByteLevel(use_regex=False)`; (3) hay tokens duplicados con los mismos bytes o un orden de merges distinto; (4) especiales. Se detecta comparando sobre una muestra con ids de ambos lados.

</details>

**4.** **(Diagnóstico)** Tu tokenizer comprime 4.2 bytes/token en español de Wikipedia pero 2.1 en tickets de soporte (jerga, números, mayúsculas). ¿Qué haces?

<details>
<summary>Ver respuesta</summary>

El vocabulario se entrenó con datos distintos del dominio: mide con una muestra del dominio, considera reentrenar o **ampliar** el vocabulario (como Llama 3 con 28 K tokens) y revisa el regex (números, mayúsculas). Ojo: cambiar el tokenizer de un modelo ya entrenado requiere adaptación (módulo 07).

</details>

**5.** **(Transferencia)** Quieres que un LLM sume números de 7 dígitos con fiabilidad. ¿Qué propiedades del pre-tokenizador y del vocabulario te importan?

<details>
<summary>Ver respuesta</summary>

Que los dígitos se partan de forma consistente (p. ej. un dígito por token o grupos alineados a la derecha), no en grupos de 3 desde la izquierda dependientes de la longitud; Singh & Strouse (2024) muestran que el sentido de la partición cambia la exactitud.

</details>

**6.** ¿Por qué el vocabulario de $V'$ tokens de un BPE es un prefijo del de $V>V'$?

<details>
<summary>Ver respuesta</summary>

Porque cada merge se elige con el corpus resultante de aplicar los anteriores: el entrenamiento a $V'$ es exactamente las primeras $V'-256$ iteraciones del entrenamiento a $V$.

</details>

**7.** ¿Qué garantiza la complejidad $O(N\log M)$ de la codificación y qué estructura la logra?

<details>
<summary>Ver respuesta</summary>

Una cola de prioridad con los pares adyacentes por rango y una lista doblemente enlazada de los tokens: cada fusión actualiza O(1) vecinos y cuesta $O(\log n)$ en el heap (Zouhar et al., 2023).

</details>

> **➡️ Siguiente módulo.** **03 · WordPiece, Unigram, SentencePiece, tiktoken y HF tokenizers.** Ya sabes construir y verificar un BPE. En el módulo 03 ves otros dos algoritmos (WordPiece y Unigram LM), la regularización por sub-palabras (BPE-dropout), el pipeline de `tokenizers` (normalizer → pre-tokenizer → model → post-processor → decoder) y comparas los tres sobre Políglota.

## 16. Referencias
Referencias de arXiv comprobadas (título y autores) el 25-oct-2026; los datos de `tiktoken` provienen de la versión 0.14.0 instalada.

* Gage, P. (1994). *A New Algorithm for Data Compression.* The C Users Journal 12(2): 23–38.
* Sennrich, R., Haddow, B., Birch, A. (2016). *Neural Machine Translation of Rare Words with Subword Units.* ACL 2016. arXiv:1508.07909. https://arxiv.org/abs/1508.07909
* Wang, C., Cho, K., Gu, J. (2019). *Neural Machine Translation with Byte-Level Subwords.* arXiv:1909.03341. https://arxiv.org/abs/1909.03341
* Radford, A., Wu, J., Child, R., Luan, D., Amodei, D., Sutskever, I. (2019). *Language Models are Unsupervised Multitask Learners* (GPT-2). OpenAI.
* Zouhar, V. et al. (2023). *A Formal Perspective on Byte-Pair Encoding.* arXiv:2306.16837. https://arxiv.org/abs/2306.16837
* Kozma, L., Voderholzer, J. (2024). *Theoretical Analysis of Byte-Pair Encoding.* arXiv:2411.08671. https://arxiv.org/abs/2411.08671
* Schmidt, C. W. et al. (2024). *Tokenization Is More Than Compression.* arXiv:2402.18376. https://arxiv.org/abs/2402.18376
* Bostrom, K., Durrett, G. (2020). *Byte Pair Encoding is Suboptimal for Language Model Pretraining.* arXiv:2004.03720. https://arxiv.org/abs/2004.03720
* Singh, A. K., Strouse, D. J. (2024). *Tokenization counts: the impact of tokenization on arithmetic in frontier LLMs.* arXiv:2402.14903. https://arxiv.org/abs/2402.14903
* Petrov, A. et al. (2023). *Language Model Tokenizers Introduce Unfairness Between Languages.* arXiv:2305.15425. https://arxiv.org/abs/2305.15425
* Ahia, O. et al. (2023). *Do All Languages Cost the Same?* arXiv:2305.13707. https://arxiv.org/abs/2305.13707
* Land, S., Bartolo, M. (2024). *Fishing for Magikarp: Automatically Detecting Under-trained Tokens in Large Language Models.* arXiv:2405.05417. https://arxiv.org/abs/2405.05417
* Llama Team, AI @ Meta (2024). *The Llama 3 Herd of Models.* arXiv:2407.21783. https://arxiv.org/abs/2407.21783
* Provilkov, I., Emelianenko, D., Voita, E. (2019). *BPE-Dropout.* arXiv:1910.13267. https://arxiv.org/abs/1910.13267
* Liu, A. et al. (2025). *SuperBPE: Space Travel for Language Models.* arXiv:2503.13423. https://arxiv.org/abs/2503.13423
* Pagnoni, A. et al. (2024). *Byte Latent Transformer.* arXiv:2412.09871. https://arxiv.org/abs/2412.09871
* Shannon, C. E. (1948). *A Mathematical Theory of Communication.* Bell System Technical Journal.
* Código: `tiktoken` (https://github.com/openai/tiktoken, incl. `tiktoken/_educational.py`); Karpathy, `minbpe` (https://github.com/karpathy/minbpe); Hugging Face `tokenizers` (https://github.com/huggingface/tokenizers); `rsennrich/subword-nmt` (https://github.com/rsennrich/subword-nmt).